# FoodScholar — Quickstart

Three cells from zero to a populated graph using your local corpus + pre-computed NER/NEL.

```
pip install -e '.[elastic,neo4j,ontology]'
# local services (matching the inline config below):
docker run -d -p 9200:9200 -e discovery.type=single-node \
    -e xpack.security.enabled=false elasticsearch:8.13.0
docker run -d -p 7687:7687 -p 7474:7474 \
    -e NEO4J_AUTH=neo4j/change-me neo4j:5
```

This notebook does **not** run GLiNER. Pre-computed annotations from
`data/foodscholar/ner/*.csv` are loaded and attached to the chunks — fast,
deterministic, no model downloads. The "Under the hood" appendix at the
bottom shows how to do the same end-to-end with GLiNER + HNSW when no
pre-computed output is on disk.

## 1. Configure

In [ ]:
import os
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC = REPO_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

# Groq key for the LLM steps (Layer-A bottom-up grouping + semantic consolidation).
# Set it in the environment BEFORE from_config so fs.llm is the real Groq client,
# not the mock. (Prefer exporting GROQ_API_KEY in your shell over hardcoding.)
os.environ.setdefault("GROQ_API_KEY", "***REMOVED-GROQ-KEY***")

from foodscholar import FoodScholar
from foodscholar.logging import configure_logging
configure_logging(level="INFO")

CORPUS_DIR = REPO_ROOT / "data" / "foodscholar" / "corpus"   # chunks_*.csv
NEL_DIR    = REPO_ROOT / "data" / "foodscholar" / "ner"      # nel_chunks_*.csv
FOODON_OWL = REPO_ROOT / "data" / "foodon.owl"

fs = FoodScholar.from_config({
    "corpus": {
        "chunks_path": str(CORPUS_DIR),
        "annotated_snapshot_path": str(REPO_ROOT / "data" / "annotated.parquet"),
    },
    "ontology": {
        "foodon_path": str(FOODON_OWL),
        "cache_path": str(REPO_ROOT / "data" / "foodon_cache.parquet"),
        "prefix_filter": ["FOODON:"],
    },
    # Real LLM for the bottom-up grouping path (group proposal + per-leaf
    # assignment) and semantic consolidation. llama-3.1-8b-instant per the
    # Layer-A rework: the grouping calls are constrained classification, cheap.
    "llm": {
        "primary": {"provider": "groq", "model": "llama-3.1-8b-instant"},
    },
    "storage": {
        "chunk_store": {
            "backend": "elastic",
            "url": "http://localhost:9200",
            "index": "foodscholar_chunks",
        },
        "graph_store": {
            "backend": "neo4j",
            "url": "bolt://localhost:7687",
            "user": "neo4j",
            "password": "password",
        },
    },
    "layer_a": {
        # --- Layer A is now built BOTTOM-UP for the foods facet ---------------
        # Instead of the old top-down prune (a flat ~186-wide blob that dropped
        # ~2,200 named foods), the foods facet starts from every corpus-mentioned
        # FoodOn leaf (coverage by construction), then an LLM groups those leaves
        # into ~14 recognizable human food categories, each anchored to a real
        # FoodOn id and shown by a friendly display label.
        # See docs/methods_layer_a_rework_brief.md.
        "facet_overrides": {
            "foods": {
                "bottom_up_grouping": {
                    "enabled": True,
                    "model": "llama-3.1-8b-instant",
                    "n_groups": 14,
                    "assign_batch_size": 60,
                    "min_leaf_support": 1,
                    # frozen_groups: leave unset to let the LLM propose the group
                    # set live; set a reviewed list for reproducible builds.
                },
            },
        },
        # link_blocklist still filters generic mentions the linker dumps onto the
        # umbrella term FOODON:00001002 ('food product') so they don't pollute
        # the per-leaf evidence. (Keeps the default 'fish' entry.)
        "link_blocklist": [
            {"surface": "fish", "ontology_id": "FOODON:00002281"},
            {"surface": "foods", "ontology_id": "FOODON:00001002"},
            {"surface": "food products", "ontology_id": "FOODON:00001002"},
            {"surface": "food product", "ontology_id": "FOODON:00001002"},
            {"surface": "whole foods", "ontology_id": "FOODON:00001002"},
            {"surface": "perishable foods", "ontology_id": "FOODON:00001002"},
            {"surface": "perishable products", "ontology_id": "FOODON:00001002"},
            {"surface": "foods and beverages", "ontology_id": "FOODON:00001002"},
            {"surface": "food and beverages", "ontology_id": "FOODON:00001002"},
            {"surface": "certain foods", "ontology_id": "FOODON:00001002"},
            {"surface": "specific food", "ontology_id": "FOODON:00001002"},
            {"surface": "real food", "ontology_id": "FOODON:00001002"},
            {"surface": "superfoods", "ontology_id": "FOODON:00001002"},
            {"surface": "imported foods", "ontology_id": "FOODON:00001002"},
            {"surface": "competitive foods", "ontology_id": "FOODON:00001002"},
            {"surface": "local foods", "ontology_id": "FOODON:00001002"},
        ],
    },
})
print("LLM:", fs.llm.model_id)  # expect llama-3.1-8b-instant, NOT mock-llm
fs.info()


## 2. Init stores & ingest

`fs.init()` creates the Elastic index and the Neo4j unique constraints — idempotent.

`fs.ingest(CORPUS_DIR, nel_dir=NEL_DIR)` reads every `chunks_*.csv`, attaches the matching annotations from `nel_*.csv`, and upserts to Elastic. **No GLiNER, no HNSW, no embedding** — this is fast and works without the `[annotate]` extra installed.

Chunk-text embeddings are populated in a separate step (§3) so you can iterate on annotations without re-encoding.

In [ ]:
fs.init()
# Abstracts are excluded by design — the prototype's `nel_*.csv` set has no
# nel_*abstracts*.csv file, so a chunk loaded from chunks_abstracts.csv would
# arrive with zero NEL annotations. Adding 600k+ such chunks would inflate the
# corpus pool without contributing any support to Layer A. Restore them only
# after the abstract NEL pass is re-run (or after switching to GLiNER+HNSW
# via `fs.load_and_annotate(CORPUS_DIR)`).
meta = fs.ingest(CORPUS_DIR, nel_dir=NEL_DIR, ignore_source_types=['abstract'])
print(meta)

## 3. Embed (optional, for vector search)

Run this once chunks are in the store and you want kNN search to work. It builds the
production embedder — `SourceTypeRouter(SPECTER2 for abstracts, BGE-large for guides /
textbooks)` — lazily on the first call (~1.7 GB model load) and patches each chunk's
`embedding` + `embedding_model` fields in Elastic via a scoped `_update` (annotations untouched).

Default `only_missing=True` skips chunks that already carry a real vector, so re-runs after
adding new chunks only encode what's new.

Requires the `[annotate]` extra (`pip install -e '.[annotate]'`).

In [ ]:
meta = fs.embed()
print(meta)

## 4. Build & explore entities

Linked entities are first-class. `fs.build_entities()` walks the chunk store, dedupes every `EntityLink` by `ontology_id` (FOODON, CHEBI, GAZ, ...), aggregates `mention_count` / `chunk_count` / sample `chunk_ids`, enriches FoodOn ids with label + synonyms + ancestors from the loaded ontology, and writes everything to:

- **Elastic** — a dedicated `foodscholar_chunks_entities` index for fast lookup + search.
- **Neo4j** — `(:Entity)` nodes plus `(:Chunk)-[:MENTIONS {confidence, method}]->(:Entity)` edges.

Then `fs.entities` is the user-facing handle: `.list(prefix="FOODON")`, `.get(id)`, `.search("olive")`, `.chunks_for(id)`.

In [ ]:
meta = fs.build_entities()
print(meta)



In [ ]:
print(f"\ntotal entities: {len(fs.entities)}")
# Top FoodOn entities by chunk_count.
for ent in fs.entities.list(k=8):
    print(f"  {ent.ontology_id:24s}  {ent.label!r:40s}  chunks={ent.chunk_count}")

# Lexical search over labels + synonyms.
print("\nsearch 'olive':")
for ent in fs.entities.search("olive", k=3):
    print(f"  {ent.ontology_id}  {ent.label}")

# Reverse lookup: chunks that mention a specific entity.
top = fs.entities.list(prefix="FOODON", k=1)
if top:
    print(f"\nchunks mentioning {top[0].ontology_id} ({top[0].label}):")
    for c in fs.entities.chunks_for(top[0].ontology_id, k=3):
        print(f"  {c.chunk_id}  {c.text[:80]}")

## 5. Build Layer A (backbone)

Layer A is FoodOn (and the other OBO ontologies we link to) projected onto **your** corpus — not a copy of the raw ontology. For every FoodOn class, count how many chunks have at least one link to that class or any of its descendants, then prune ruthlessly: blacklist organizational classes, drop everything below `min_support`, lift shelves above `max_depth` to a closer ancestor, and collapse single-child chains into their leaves (recording the dropped ids in `see_also`).

Non-foods facets (`health`, `nutrients`, `dietary_patterns`, `allergies`) derive from `Mention.entity_type` on EntityLinks. The prototype's pre-computed NEL data sets every `entity_type` to `"other"`, so on this corpus those facets land as **stub roots only** — they'll fill in once chunks are re-annotated with GLiNER. Sustainability is always a stub root (no entity_type maps to it).

Layer A here is **projection only**. Chunk attachment (`fs.attach()`) is the next phase.

Re-running `fs.build_layer_a()` is idempotent: every call clears the prior `:Shelf` projection from the graph store before writing the new one, so you can re-tune `min_support` / `blacklist_terms` / `facet_overrides` freely without stale ghost shelves piling up across iterations.

In [ ]:
# Layer A build.
#
# FOODS facet → bottom-up LLM grouping (configured in §1 via facet_overrides):
#   leaves(all mentioned foods) → LLM proposes ~14 groups → assign each leaf to a
#   group by label → flat group shelves (display_label) + kept-leaf shelves for
#   the unassigned. The top-down prune knobs below (min_support / max_depth /
#   umbrella / blacklist) DO NOT apply to foods anymore — they only affect the
#   other facets that still use the top-down path.
#
# OTHER facets (health / nutrients / …) still use top-down prune; tune them here.
fs.config.layer_a.min_support = 25
fs.config.layer_a.max_depth = 6
fs.config.layer_a.collapse_single_child_chains = True
fs.config.layer_a.blacklist_terms = fs.config.layer_a.blacklist_terms + [
    "food calorie datum",
    "edible food",
]

# Echo the resolved foods-facet config so the active path is visible at a glance.
_foods = fs.config.layer_a.resolve_facet('foods')
_bu = _foods.bottom_up_grouping
if _bu.enabled:
    print(f"foods: BOTTOM-UP grouping ON — model={_bu.model}, n_groups={_bu.n_groups}, "
          f"assign_batch_size={_bu.assign_batch_size}, min_leaf_support={_bu.min_leaf_support}, "
          f"frozen_groups={'yes' if _bu.frozen_groups else 'no (live LLM proposal)'}")
    if fs.llm.model_id.startswith("mock"):
        print("  ⚠️  fs.llm is the MOCK client — set GROQ_API_KEY (see §1) or the foods "
              "facet will degrade to one flat shelf per leaf (the un-navigable blob).")
else:
    print(f"foods: top-down prune — min_support={_foods.min_support}, max_depth={_foods.max_depth}")

meta = fs.build_layer_a()
print(meta)

# Quick shape check: how many group shelves vs kept-leaf shelves did foods get?
from collections import Counter
foods_shelves = [s for s in fs.graph.shelves(facet='foods')]
n_groups = sum(1 for s in foods_shelves if (s.model.display_label and s.model.see_also))
n_leaves = sum(1 for s in foods_shelves if (s.model.display_label and not s.model.see_also))
print(f"foods shelves: {len(foods_shelves)} total — {n_groups} group shelves, "
      f"{n_leaves} kept-leaf shelves")


In [ ]:
from collections import Counter

shelves = fs.graph_store.list_shelves()
by_facet = Counter(s.facet for s in shelves)
print('shelves per facet:')
for facet, n in sorted(by_facet.items(), key=lambda kv: -kv[1]):
    print(f'  {facet:18s} {n}')

foods = [s for s in shelves if s.facet == 'foods']
print(f'\ntop 10 foods shelves by chunk_count:')
for s in sorted(foods, key=lambda s: -s.chunk_count)[:10]:
    direct_share = s.support_direct / s.chunk_count if s.chunk_count else 0
    print(f'  {s.label[:40]:40s}  {s.chunk_count:>5d}  (direct={s.support_direct} lifted={s.support_lifted} direct_share={direct_share:.3f})')

print(f'\nbottom 5 surviving foods shelves:')
for s in sorted(foods, key=lambda s: s.chunk_count)[:5]:
    print(f'  {s.label[:40]:40s}  {s.chunk_count:>5d}')

depths = Counter(s.depth for s in foods)
print(f'\nfoods depth distribution:')
for d in sorted(depths):
    print(f'  depth {d}: {depths[d]}')

# Inflation diagnostic — shelves whose support is mostly lifted (per the spec,
# >0.9 means a deep dense subtree funneled into one ancestor). After the
# umbrella rule lands these should be rare — surviving inflated shelves are
# real navigation roots (e.g. `food product` itself) rather than scaffolding.
inflated = [s for s in foods if s.chunk_count > 0 and s.support_lifted / s.chunk_count > 0.9]
if inflated:
    print(f'\n{len(inflated)} potentially-inflated shelves (support_lifted/chunk_count > 0.9):')
    for s in sorted(inflated, key=lambda s: -s.chunk_count)[:5]:
        ratio = s.support_lifted / s.chunk_count
        direct_share = s.support_direct / s.chunk_count
        print(f'  {s.label[:40]:40s}  lifted_ratio={ratio:.2f}  direct_share={direct_share:.3f}  count={s.chunk_count}')
else:
    print('\nno inflation flags.')

# Orphan diagnostic — shelves at depth 0 that aren't the stub roots. After
# the umbrella rule lands these are the navigation roots that no surviving
# parent could be found for. Whitelisting the foods root (FOODON:00001002)
# typically reduces this to 1-2.
depth0_foods = [s for s in foods if s.depth == 0]
print(f'\n{len(depth0_foods)} foods shelves at depth 0 (roots):')
for s in sorted(depth0_foods, key=lambda s: -s.chunk_count)[:10]:
    print(f'  {s.label[:40]:40s}  count={s.chunk_count}  foodon_id={s.foodon_id}')

  [food            ] 'olive oil'  @34:43  (keyword-ner-v0)
  [food            ] 'whole'  @45:50  (keyword-ner-v0)


### 5b. Why these thresholds? — parameter sweep

The three knobs the prune cascade exposes (`min_support`, `umbrella_direct_share_max`, `umbrella_min_count`) are empirical, not derived. This cell sweeps each one independently against the live corpus and plots the trade-off curves, then a 2D heatmap of the two umbrella knobs together. The chosen values are marked.

Cheap: support is collected once over the chunk store, then `prune(...)` re-runs per config combo without touching Neo4j. Runs in seconds even on a multi-thousand-chunk corpus.

In [ ]:
import matplotlib.pyplot as plt
from foodscholar.layer_a.propagate import collect_support
from foodscholar.layer_a.prune import prune
from foodscholar.config import FacetConfig, LayerAConfig

# Collect support once — every sweep point re-prunes against this in-memory
# table. ~seconds for ~13k chunks; the bottleneck would be Neo4j writes,
# which we skip.
def _chunk_iter():
    for batch in fs.chunk_store.iter_chunks():
        yield from batch

base = fs.config.layer_a.resolve_facet('foods')
support = collect_support(_chunk_iter(), fs.ontology,
                          min_link_confidence=base.min_link_confidence,
                          facet='foods')
print(f'support table: {len(support.with_descendants)} terms, '
      f'{sum(support.direct.values())} direct mentions')

def _resolve_with(**overrides):
    """Build a fully-resolved foods FacetConfig by overlaying overrides on the
    live `fs.config.layer_a`. Goes through `resolve_facet` so we touch only the
    public config surface."""
    base_cfg = fs.config.layer_a
    layer_a = LayerAConfig.model_validate(base_cfg.model_dump())
    layer_a.facet_overrides = dict(layer_a.facet_overrides)
    layer_a.facet_overrides['foods'] = FacetConfig(**overrides)
    return layer_a.resolve_facet('foods')

def _summarize(shelves: list) -> dict:
    inflated = sum(1 for s in shelves
                   if s.chunk_count > 0 and s.support_lifted / s.chunk_count > 0.9)
    roots = sum(1 for s in shelves if s.parent_shelf_id is None)
    return {'n_shelves': len(shelves), 'n_inflated': inflated, 'n_roots': roots}

# Run the three single-knob sweeps.
ms_grid = [5, 10, 15, 20, 25, 30, 40, 50, 75, 100, 150, 200]
ds_grid = [0.0, 0.03, 0.05, 0.075, 0.10, 0.125, 0.15, 0.20, 0.30, 0.50]
mc_grid = [10, 25, 50, 75, 100, 150, 200, 300, 500, 1000]

ms_data = [(ms, _summarize(prune(support, fs.ontology, _resolve_with(min_support=ms), 'foods'))) for ms in ms_grid]
ds_data = [(ds, _summarize(prune(support, fs.ontology, _resolve_with(umbrella_direct_share_max=ds), 'foods'))) for ds in ds_grid]
mc_data = [(mc, _summarize(prune(support, fs.ontology, _resolve_with(umbrella_min_count=mc), 'foods'))) for mc in mc_grid]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

def _plot(ax, data, knob_name, chosen, log_x=False):
    xs = [d[0] for d in data]
    n_shelves = [d[1]['n_shelves'] for d in data]
    n_inflated = [d[1]['n_inflated'] for d in data]
    ax.plot(xs, n_shelves, 'o-', label='surviving shelves')
    ax.plot(xs, n_inflated, 's--', label='inflated (>0.9 lifted)')
    ax.axvline(chosen, linestyle=':', label=f'chosen = {chosen}')
    ax.set_title(knob_name)
    ax.set_xlabel(knob_name)
    ax.set_ylabel('shelves')
    ax.grid(True)
    if log_x:
        ax.set_xscale('log')
    ax.legend()

_plot(axes[0], ms_data, 'min_support', base.min_support, log_x=True)
_plot(axes[1], ds_data, 'umbrella_direct_share_max', base.umbrella_direct_share_max)
_plot(axes[2], mc_data, 'umbrella_min_count', base.umbrella_min_count, log_x=True)

fig.suptitle('Layer A — single-knob parameter sweep')
plt.tight_layout()
plt.show()

# Print the chosen-point summary as numeric grounding for the chart.
chosen_cfg = _resolve_with()  # all defaults
chosen_shelves = prune(support, fs.ontology, chosen_cfg, 'foods')
chosen = _summarize(chosen_shelves)
print(f"\nat chosen config (min_support={base.min_support}, "
      f"direct_share_max={base.umbrella_direct_share_max}, "
      f"min_count={base.umbrella_min_count}):")
print(f"  shelves:               {chosen['n_shelves']}")
print(f"  inflated (>0.9 lifted): {chosen['n_inflated']}")
print(f"  orphan roots:           {chosen['n_roots']}")

ontology.cache_hit path=/mnt/workspaces/wisefood/foodscholar-lib/data/foodon_cache.parquet
HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-19T12:46:51.461720Z [info     ] agent_ner.extracted            model=llama-3.3-70b-versatile n_kept=3 n_returned=3


c1: Mediterranean diet rich in olive oil reduces cardiovascular risk.
    [dietary_pattern ] 'Mediterranean diet'  @0:18
    [food            ] 'olive oil'  @27:36
    [biomarker       ] 'cardiovascular risk'  @45:64


HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-19T12:46:51.988214Z [info     ] agent_ner.extracted            model=llama-3.3-70b-versatile n_kept=2 n_returned=2


c2: Whole grain consumption is associated with lower mortality.
    [processing      ] 'Whole grain'  @0:11
    [biomarker       ] 'mortality'  @49:58


HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-19T12:46:52.688701Z [info     ] agent_ner.extracted            model=llama-3.3-70b-versatile n_kept=2 n_returned=2


...

c4: Dietary intake of quinoa improves glycemic control in adults with type 2 diabetes.
    [food            ] 'quinoa'  @18:24
    [biomarker       ] 'glycemic control'  @34:50
    [population      ] 'adults'  @54:60
    [health          ] 'type 2 diabetes'  @66:81


HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-19T12:46:53.584325Z [warning  ] agent_ner.mention_not_in_text  mention='extra virgin'
2026-05-19T12:46:53.598713Z [info     ] agent_ner.extracted            model=llama-3.3-70b-versatile n_kept=3 n_returned=4


c5: Extra virgin olive oil contains polyphenols that reduce inflammation markers.
    [food            ] 'Extra virgin olive oil'  @0:22
    [nutrient        ] 'polyphenols'  @32:43
    [biomarker       ] 'inflammation markers'  @56:76


HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-19T12:46:53.927564Z [info     ] agent_ner.extracted            model=llama-3.3-70b-versatile n_kept=2 n_returned=2


c6: School-based peanut allergy education reduces accidental exposures among children.
    [allergen        ] 'peanut allergy'  @13:27
    [population      ] 'children'  @73:81


In [ ]:
import numpy as np

# Joint sweep: direct_share_max × min_count. Heatmap colors the joint effect.
ds_axis = [0.03, 0.05, 0.075, 0.10, 0.125, 0.15, 0.20, 0.30]
mc_axis = [25, 50, 75, 100, 150, 200, 300, 500]

grid_shelves = np.zeros((len(mc_axis), len(ds_axis)))
grid_inflated = np.zeros((len(mc_axis), len(ds_axis)))
for i, mc in enumerate(mc_axis):
    for j, ds in enumerate(ds_axis):
        cfg = _resolve_with(umbrella_direct_share_max=ds, umbrella_min_count=mc)
        shelves = prune(support, fs.ontology, cfg, 'foods')
        s = _summarize(shelves)
        grid_shelves[i, j] = s['n_shelves']
        grid_inflated[i, j] = s['n_inflated']
clean = grid_shelves - grid_inflated

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

def _heatmap(ax, grid, cmap, title):
    im = ax.imshow(grid, aspect='auto', origin='lower', cmap=cmap)
    ax.set_xticks(range(len(ds_axis)))
    ax.set_xticklabels([f'{d:g}' for d in ds_axis])
    ax.set_yticks(range(len(mc_axis)))
    ax.set_yticklabels(mc_axis)
    ax.set_xlabel('umbrella_direct_share_max')
    ax.set_ylabel('umbrella_min_count')
    ax.set_title(title)
    fig.colorbar(im, ax=ax)
    for i in range(len(mc_axis)):
        for j in range(len(ds_axis)):
            ax.text(j, i, f'{grid[i, j]:.0f}', ha='center', va='center')

_heatmap(axes[0], clean, 'viridis', 'Clean shelves (surviving − inflated)')
_heatmap(axes[1], grid_inflated, 'Reds', 'Inflated shelves')

# Mark the chosen config in both heatmaps.
if base.umbrella_direct_share_max in ds_axis and base.umbrella_min_count in mc_axis:
    j_chosen = ds_axis.index(base.umbrella_direct_share_max)
    i_chosen = mc_axis.index(base.umbrella_min_count)
    for ax in axes:
        ax.add_patch(plt.Rectangle((j_chosen - 0.5, i_chosen - 0.5), 1, 1,
                                   fill=False, edgecolor='black', linewidth=2))

fig.suptitle('Layer A — joint sweep of the two umbrella knobs')
plt.tight_layout()
plt.show()

### 5c. What happens when you raise `max_depth`?

The depth cap doesn't drop deep terms — it **lifts** them to the nearest surviving ancestor at depth ≤ cap (see [prune.py](../src/foodscholar/layer_a/prune.py) and the §7.0 PROGRESS entry on op order). So raising the cap doesn't *add* terms; it lets terms that were already there land deeper instead of bunching up at the cap.

Three things should shift as the cap rises:

1. **Surviving shelves go up** — more terms keep their natural ontology depth instead of getting clipped + collapsed away.
2. **`support_lifted` redistributes downward** — chunks that were lifting onto a shallow ancestor now find a more specific match closer to the leaf. Inflation drops on shelves that were absorbing everything from below.
3. **Median direct-share rises** — deep shelves get their own chunks instead of letting them lift up.

This sweep is **projection-only** (same recipe as §5b): support is collected once, `prune()` re-runs per cap value, no Neo4j writes. Re-running `fs.attach()` per cap would also redistribute *edges*, but the projection-side picture is cheap and tells you whether raising the cap is even worth doing.

In [ ]:
from statistics import median

# Depth sweep — `support` and `_resolve_with` are still in scope from §5b.
depth_grid = [3, 4, 5, 6, 7, 8, 9]

def _depth_summary(shelves):
    if not shelves:
        return {'n_shelves': 0, 'n_inflated': 0, 'median_direct_share': 0.0,
                'depth_dist': {}, 'shelves_at_or_above_cap': 0}
    inflated = sum(1 for s in shelves
                   if s.chunk_count > 0 and s.support_lifted / s.chunk_count > 0.9)
    direct_shares = [s.support_direct / s.chunk_count
                     for s in shelves if s.chunk_count > 0]
    depth_dist = Counter(s.depth for s in shelves)
    cap = max(depth_dist) if depth_dist else 0
    return {
        'n_shelves': len(shelves),
        'n_inflated': inflated,
        'median_direct_share': median(direct_shares) if direct_shares else 0.0,
        'depth_dist': dict(depth_dist),
        'shelves_at_or_above_cap': depth_dist.get(cap, 0),
    }

depth_data = []
for d in depth_grid:
    cfg = _resolve_with(max_depth=d)
    shelves = prune(support, fs.ontology, cfg, 'foods')
    depth_data.append((d, _depth_summary(shelves)))

xs = [d[0] for d in depth_data]
n_shelves = [d[1]['n_shelves'] for d in depth_data]
n_inflated = [d[1]['n_inflated'] for d in depth_data]
median_share = [d[1]['median_direct_share'] for d in depth_data]

CHOSEN_DEPTH = fs.config.layer_a.max_depth

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# Panel 1: shelves + inflation vs depth cap.
ax = axes[0]
ax.plot(xs, n_shelves, 'o-', label='surviving shelves')
ax.plot(xs, n_inflated, 's--', label='inflated (>0.9 lifted)')
ax.axvline(CHOSEN_DEPTH, linestyle=':', label=f'chosen = {CHOSEN_DEPTH}')
ax.set_title('max_depth: surviving + inflated shelves')
ax.set_xlabel('max_depth')
ax.set_ylabel('shelves')
ax.grid(True)
ax.legend()

# Panel 2: median direct-share vs depth cap.
ax = axes[1]
ax.plot(xs, median_share, 'o-', label='median direct-share')
ax.axvline(CHOSEN_DEPTH, linestyle=':', label=f'chosen = {CHOSEN_DEPTH}')
ax.set_title('median direct-share')
ax.set_xlabel('max_depth')
ax.set_ylabel('median support_direct / chunk_count')
ax.set_ylim(0, 1.05)
ax.grid(True)
ax.legend()

# Panel 3: depth distribution as stacked bars.
ax = axes[2]
all_depths = sorted({d for _, s in depth_data for d in s['depth_dist']})
bottoms = [0] * len(xs)
for depth in all_depths:
    heights = [s[1]['depth_dist'].get(depth, 0) for s in depth_data]
    ax.bar(xs, heights, bottom=bottoms, label=f'depth {depth}')
    bottoms = [b + h for b, h in zip(bottoms, heights, strict=True)]
ax.axvline(CHOSEN_DEPTH, linestyle=':', label=f'chosen = {CHOSEN_DEPTH}')
ax.set_title('shelf count by projection depth')
ax.set_xlabel('max_depth')
ax.set_ylabel('shelves')
ax.grid(True)
ax.legend(title='projection depth', ncol=2)

fig.suptitle('Layer A — depth-cap sweep')
plt.tight_layout()
plt.show()

# Numeric grounding — print the table so the chart values are auditable.
print(f'\n{"depth":>5s}  {"shelves":>7s}  {"inflated":>8s}  {"med_direct":>10s}  '
      f'{"shelves_at_cap":>14s}')
for d, s in depth_data:
    mark = '  *' if d == CHOSEN_DEPTH else ''
    print(f'{d:>5d}  {s["n_shelves"]:>7d}  {s["n_inflated"]:>8d}  '
          f'{s["median_direct_share"]:>10.3f}  {s["shelves_at_or_above_cap"]:>14d}{mark}')

### 5d. Why does so much support land at the synthetic root? — NEL coverage audit

The synthetic `Foods` root carries `support_lifted = chunk_count` by construction — every chunk reaching the facet contributes through lifting since nothing in any ontology resolves to the root itself. But the *size* of that number relative to the total corpus is informative: a high root count means many chunks failed to find a specific shelf, either because they have no FOODON links at all, or because all their links got umbrella-pruned.

This cell walks the chunk store once and reports:

- **NEL coverage**: how many chunks have ≥1 `foodon_id`, how many have only non-FOODON `entity_links`, how many have no links at all.
- **Sample of zero-foodon chunks**: 5 random texts so you can eyeball whether they're nutrition-y-but-not-food (expected) or food-content-the-NEL-missed (a real coverage gap).

In [ ]:
import random
from collections import Counter

n_total = 0
n_with_foodon = 0
n_only_other_obo = 0  # has entity_links but zero FOODON ids
n_no_links = 0
prefix_counter: Counter = Counter()
zero_foodon_samples: list = []

for batch in fs.chunk_store.iter_chunks():
    for chunk in batch:
        n_total += 1
        has_foodon = bool(chunk.foodon_ids)
        has_any_link = bool(chunk.entity_links)
        if has_foodon:
            n_with_foodon += 1
        elif has_any_link:
            n_only_other_obo += 1
            for link in chunk.entity_links:
                oid = link.ontology_id
                prefix = oid.split(':', 1)[0] if ':' in oid else '(none)'
                prefix_counter[prefix] += 1
        else:
            n_no_links += 1

        # Keep a reservoir sample of zero-foodon chunks for hand inspection.
        if not has_foodon and len(zero_foodon_samples) < 200:
            zero_foodon_samples.append(chunk)

print(f'chunks total:                  {n_total:>6d}  (100.0%)')
print(f'  ≥1 FOODON id (well-placed):  {n_with_foodon:>6d}  ({100*n_with_foodon/n_total:>5.1f}%)')
print(f'  only non-FOODON entity_links: {n_only_other_obo:>6d}  ({100*n_only_other_obo/n_total:>5.1f}%)')
print(f'  zero entity links at all:     {n_no_links:>6d}  ({100*n_no_links/n_total:>5.1f}%)')

print(f'\nontology prefixes seen on chunks WITHOUT a FOODON id ({len(prefix_counter)} distinct):')
for prefix, n in prefix_counter.most_common(10):
    print(f'  {prefix:15s}  {n:>6d}')

print(f'\n5 random zero-foodon chunk texts (hand-check whether they should have had a FOODON link):')
random.seed(0)
for chunk in random.sample(zero_foodon_samples, k=min(5, len(zero_foodon_samples))):
    has_other = bool(chunk.entity_links)
    marker = '  [has non-FOODON links]' if has_other else '  [no links at all]'
    print(f'\n  {chunk.chunk_id}{marker}')
    print(f'    {chunk.text[:240].replace(chr(10), " ")}{"..." if len(chunk.text) > 240 else ""}')

### 5e. Sanity gate — 50 random FOODON-linked chunks

Pick 50 random chunks that have ≥1 FOODON link and print the text + the projected shelf labels each one rolls up to. The shelves shown mirror what `fs.attach()` writes in §6 — for every `foodon_id` on the chunk, this finds the deepest surviving shelf whose `foodon_id` is that term OR a surviving ancestor of it. Run this audit *before* `fs.attach()` to catch projection bugs early; run it again after to spot any drift between the resolver here and the real one.

Read 5–10 of these by hand. The question to answer: **is each chunk genuinely about the shelf it lands on?** Common failure modes to look for:
- NEL drift: a chunk mentioning "milky way" got linked to `cow milk`.
- Over-broad attachment: a chunk explicitly about kale ends up on `vegetable` instead of `kale` because the more specific shelf got pruned.
- Off-topic match: text is about cardiology but linked to `fat` via "fat embolism".

In [ ]:
import random

# Index surviving shelves three ways:
#   shelf_by_foodon  : direct foodon_id → shelf  (chunk linked to this term)
#   shelf_by_seealso : collapsed foodon_id → shelf  (term collapsed into
#                                                    survivor; see_also)
# A chunk's mention of a collapsed term legitimately attaches to the
# survivor — same as a lifted ancestor walk, just recorded differently.
shelves = fs.graph_store.list_shelves()
shelf_by_foodon = {s.foodon_id: s for s in shelves if s.foodon_id is not None}
shelf_by_seealso = {fid: s for s in shelves for fid in s.see_also}

def project_chunk(foodon_ids):
    """Returns (attached_shelves, attachment_details) where details says how
    each foodon_id reached its shelf: 'direct' / 'collapsed' / 'lifted' / None.
    Mirrors what fs.attach() will compute once it lands."""
    attached = set()
    details = []  # list of (foodon_id, label, mode, shelf_label_or_none)
    for fid in foodon_ids:
        label = fs.ontology.id_to_label(fid) or fid
        # 1. direct: shelf with this foodon_id
        if fid in shelf_by_foodon:
            sh = shelf_by_foodon[fid]
            attached.add(sh.label)
            details.append((fid, label, 'direct', sh.label))
            continue
        # 2. collapsed: term was absorbed into a surviving descendant via see_also
        if fid in shelf_by_seealso:
            sh = shelf_by_seealso[fid]
            attached.add(sh.label)
            details.append((fid, label, 'collapsed', sh.label))
            continue
        # 3. lifted: deepest surviving ancestor (umbrella/blacklist drops left a gap)
        ancestors = fs.ontology.id_to_ancestors(fid) if fs.ontology else []
        best = None
        best_depth = -1
        for anc in ancestors:
            sh = shelf_by_foodon.get(anc) or shelf_by_seealso.get(anc)
            if sh is not None and sh.depth > best_depth:
                best = sh
                best_depth = sh.depth
        if best is not None:
            attached.add(best.label)
            details.append((fid, label, 'lifted', best.label))
        else:
            details.append((fid, label, 'orphan', None))
    return sorted(attached), details

# Collect chunks with ≥1 FOODON id.
candidates = []
for batch in fs.chunk_store.iter_chunks():
    for chunk in batch:
        if chunk.foodon_ids:
            candidates.append(chunk)
        if len(candidates) >= 5000:  # cap upstream sampling cost
            break
    if len(candidates) >= 5000:
        break

# Corpus-wide counts: how many chunks with foodon_ids end up orphaned?
# (This is the systemic-bug check the audit asked for.)
n_orphaned = 0
n_collapsed_only = 0
n_lifted_only = 0
for chunk in candidates:
    _, details = project_chunk(chunk.foodon_ids)
    modes = {d[2] for d in details}
    if modes == {'orphan'}:
        n_orphaned += 1
    elif 'direct' not in modes and 'collapsed' in modes and 'lifted' not in modes:
        n_collapsed_only += 1
    elif 'direct' not in modes and 'lifted' in modes and 'collapsed' not in modes:
        n_lifted_only += 1

print(f'corpus-wide attachment audit on {len(candidates)} FOODON-linked chunks:')
print(f'  fully orphaned (no shelf):           {n_orphaned}  ← real bugs')
print(f'  reached shelf only via collapse:     {n_collapsed_only}')
print(f'  reached shelf only via lifting:      {n_lifted_only}')
if n_orphaned > 50:
    print(f'  ⚠ {n_orphaned} orphaned chunks is high; investigate systematic gap')
elif n_orphaned > 0:
    print(f'  ({n_orphaned} orphans likely from terms whose ancestry exits FOODON via non-food OBOs)')

# Sample 50 + show 10 with attachment annotations.
random.seed(0)
sample = random.sample(candidates, k=min(50, len(candidates)))
print(f'\n--- sample of {min(10, len(sample))} chunks (raise [:10] to [:50] for full audit) ---\n')
for i, chunk in enumerate(sample[:10]):
    attached, details = project_chunk(chunk.foodon_ids)
    # Group details by mode for compact display.
    direct_lines  = [f'{lbl!r}' for _, lbl, mode, _ in details if mode == 'direct']
    collapsed_lines = [f'{lbl!r} → {target}' for _, lbl, mode, target in details if mode == 'collapsed']
    lifted_lines = [f'{lbl!r} → {target}' for _, lbl, mode, target in details if mode == 'lifted']
    orphan_lines = [f'{lbl!r}' for _, lbl, mode, _ in details if mode == 'orphan']

    print(f'[{i+1}] {chunk.chunk_id}')
    print(f'    attached shelves:      {attached}')
    if direct_lines:
        print(f'    foodon → direct:       {direct_lines}')
    if collapsed_lines:
        print(f'    foodon → collapsed:    {collapsed_lines}')
    if lifted_lines:
        print(f'    foodon → lifted:       {lifted_lines}')
    if orphan_lines:
        print(f'    foodon → orphan:       {orphan_lines}  ← cannot reach any shelf')
    text = chunk.text.replace('\n', ' ')
    print(f'    text: {text[:220]}{"..." if len(text) > 220 else ""}')
    print()

# Drift candidates: (surface, ontology_id) pairs across the sample that
# repeat ≥ 3 times. These are candidates to add to
# `fs.config.layer_a.link_blocklist` if the audit shows they're
# semantically wrong. Surface forms are case-insensitive in the blocklist.
from collections import Counter
drift_counter: Counter = Counter()
for chunk in sample:
    for link in chunk.entity_links:
        if link.ontology_id.startswith('FOODON:'):
            drift_counter[(link.mention.text.lower(), link.ontology_id)] += 1

frequent = [(pair, n) for pair, n in drift_counter.most_common() if n >= 3]
if frequent:
    print(f'\n--- (surface, ontology_id) pairs appearing ≥3 times across {len(sample)} sampled chunks ---')
    print('add to link_blocklist if semantically wrong:')
    for (surface, oid), n in frequent[:15]:
        label = fs.ontology.id_to_label(oid) or oid
        print(f'  {n:>3d}× ({surface!r:>30}, {oid:25s}) → label={label!r}')
else:
    print('\nno (surface, ontology_id) pair repeats ≥3 times — no obvious drift in this sample.')


### 5f. Sanity gate — 20 chunks with non-FOODON links only

These are the chunks that reached the foods facet only via the synthetic root (no FOODON id, but ≥1 entity_link of some other prefix). The question: **does the text contain a food term that should have linked but didn't?** That's a NEL coverage gap — the upstream linker missed a food mention.

If 5+ of 20 sampled chunks have an obvious missed food term, re-annotation with GLiNER is the right next step. If most are nutrition-y-but-not-food (papers about "macronutrient distribution", clinical population studies, etc.), the synthetic-root lift is expected and harmless.

In [ ]:
candidates_nonfoodon = []
for batch in fs.chunk_store.iter_chunks():
    for chunk in batch:
        if not chunk.foodon_ids and chunk.entity_links:
            candidates_nonfoodon.append(chunk)
        if len(candidates_nonfoodon) >= 5000:
            break
    if len(candidates_nonfoodon) >= 5000:
        break

random.seed(1)
sample_nf = random.sample(candidates_nonfoodon, k=min(20, len(candidates_nonfoodon)))
print(f'sampled {len(sample_nf)} of {len(candidates_nonfoodon)} non-FOODON-only chunks scanned\n')

for i, chunk in enumerate(sample_nf):
    surface_forms = [link.mention.text for link in chunk.entity_links[:5]]
    prefixes = sorted({link.ontology_id.split(':', 1)[0] for link in chunk.entity_links})
    print(f'[{i+1}] {chunk.chunk_id}')
    print(f'    linked mentions ({len(chunk.entity_links)}): {surface_forms}{" ..." if len(chunk.entity_links) > 5 else ""}')
    print(f'    prefixes seen: {prefixes}')
    text = chunk.text.replace("\n", " ")
    print(f'    text: {text[:260]}{"..." if len(text) > 260 else ""}')
    print()

## 6. Attach chunks to shelves

`fs.attach()` is the bridge phase between Layer A projection and everything downstream. It walks every chunk, resolves each FOODON id to a surviving shelf via the same direct → collapsed → lifted cascade the §5d audit previews, then writes:

- **Neo4j**: `(:Chunk)-[:ATTACHED_TO {lifted_from: [foodon_id,...]}]->(:Shelf)` edges. `lifted_from` is empty for direct attachments (chunk linked the shelf's own term); non-empty when the chunk reached the shelf via collapse or by lifting through ancestors.
- **Elastic**: each chunk's `shelf_ids` keyword-array gets the union of its resolved shelves, so retrieval can filter `terms { shelf_ids: [...] }` without crossing the graph.

Attachment is **nearest surviving ancestor only** (one shelf per FOODON id, the deepest match), facet-aware (`route_link_to_facet` keeps a `medical condition` mention out of the foods walk), and idempotent (re-running with the same projection produces the same edges and the same `lifted_from` payloads). Orphan FOODON ids whose ancestry can't reach any real shelf fall through to the synthetic facet root (`facet:foods`, etc.).

In [ ]:
meta = fs.attach()
print(meta)

# Per-shelf attachment counts straight from Neo4j. Order matches the §5d
# audit's "top 10 foods shelves by chunk_count" so any discrepancy between
# projection-time `chunk_count` and post-attach edge count flags drift —
# typically a chunk whose deepest surviving ancestor differs from the term
# that drove support during projection (e.g. when collapse rewrites the
# parent chain after support was already counted).
from collections import Counter

attach_counts: Counter = Counter()
lifted_edges = 0
direct_edges = 0
with fs.graph_store._driver.session() as session:  # type: ignore[attr-defined]
    result = session.run(
        """
        MATCH (c:Chunk)-[r:ATTACHED_TO]->(s:Shelf)
        WHERE s.facet = 'foods'
        RETURN s.shelf_id AS sid, s.label AS label,
               count(*) AS n,
               sum(CASE WHEN size(r.lifted_from) = 0 THEN 1 ELSE 0 END) AS direct,
               sum(CASE WHEN size(r.lifted_from) > 0 THEN 1 ELSE 0 END) AS lifted
        ORDER BY n DESC
        LIMIT 12
        """
    )
    print(f'\ntop foods shelves by attached chunks (graph edges):')
    for row in result:
        attach_counts[row['sid']] = row['n']
        direct_edges += row['direct']
        lifted_edges += row['lifted']
        print(f'  {row["label"][:40]:40s}  {row["n"]:>5d}  '
              f'(direct={row["direct"]}, via lifted_from={row["lifted"]})')

print(f'\nedge breakdown in top 12: direct={direct_edges}, lifted/collapsed={lifted_edges}')

# Pick one chunk and show what it landed on — concrete confirmation that
# the in-Elastic shelf_ids denorm matches the graph edges, and that
# lifted_from payloads survive the round-trip.
sample_chunk = next(
    (c for c in fs.chunk_store.scan() if c.shelf_ids), None
)
if sample_chunk is not None:
    print(f'\nsample chunk {sample_chunk.chunk_id!r}:')
    print(f'  foodon_ids on chunk:    {sample_chunk.foodon_ids[:6]}')
    print(f'  shelf_ids (Elastic):    {sample_chunk.shelf_ids}')
    with fs.graph_store._driver.session() as session:  # type: ignore[attr-defined]
        result = session.run(
            """
            MATCH (c:Chunk {chunk_id: $cid})-[r:ATTACHED_TO]->(s:Shelf)
            RETURN s.label AS label, s.shelf_id AS sid, r.lifted_from AS lifted_from
            """,
            cid=sample_chunk.chunk_id,
        )
        print(f'  Neo4j edges:')
        for row in result:
            mode = 'direct' if not row['lifted_from'] else f'lifted_from={row["lifted_from"]}'
            print(f'    -> {row["label"]!r:40s}  ({mode})')

## 6b. Audit + quality report

Two checks, two different questions:

- **`fs.audit()`** — *is the graph correctly built?* Five invariants across the chunk store, entity store, and graph store. Critical failures mean Layer B will build on a broken Layer A. Use this to gate downstream work.
- **`fs.quality_report(facet='foods')`** — *is the graph good?* A domain-expert-facing report with five sections (top shelves, hierarchy walkthrough, suspicious shelves, canonical vocabulary check, random chunk sample) formatted as Markdown for in-notebook review.

Audit answers yes/no with hard thresholds; quality is a conversation a nutritionist reads top-to-bottom in ~30 minutes. The quality report is intentionally NOT a single score — quality is multidimensional, and collapsing it would hide the failures that matter.

In [ ]:
# Audit first — invariants. If this fails, the rest is on shaky ground.
audit = fs.audit()
print(audit)
print()
if audit.critical_failures:
    print(f'⚠ {len(audit.critical_failures)} critical invariant(s) failing. '
          'Inspect audit.critical_failures[i].sample for examples.')

# Then quality. Long Markdown — render via IPython for in-notebook readability.
from IPython.display import Markdown

quality = fs.quality_report(facet='foods', top_n=15, sample_size=20, seed=0)
Markdown(str(quality))

### 6c. Layer A connectivity profile

The audit answers *is the graph correctly wired?* with yes/no invariants; this cell answers
*what does the wiring **look like**?* — the quantitative shape of Layer A's connectivity, the
diagnostics you'd skim before trusting downstream Layer B clustering.

Two graphs make up Layer A's connectivity and we profile both:

- **Shelf tree** (`(:Shelf)-[parent]->(:Shelf)`) — the navigation backbone. We read it
  straight off `parent_shelf_id` so this works on *any* backend (Neo4j or in-memory). Panels
  cover **branching factor** (children per shelf), **depth profile per facet**, and the
  **support-weight CCDF** (how chunk_count concentrates across shelves).
- **Chunk→shelf bipartite layer** (`(:Chunk)-[:ATTACHED_TO]->(:Shelf)`) — the actual
  attachment fan-out. Pulled from Neo4j edges when the graph store exposes a driver; the panel
  is skipped with a note on backends that don't (the projection-side `chunk_count` already
  covers the same ground in §5).

Read it as: a healthy backbone is a *tree* (mean branching 2–5, few orphan roots), support is
**heavy-tailed but not single-peaked** (a handful of broad shelves + a long body of specific
ones), and the synthetic facet root absorbs a *minority* of attachments (a majority means NEL
coverage is thin — see the §5d audit).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter, defaultdict

FACET = 'foods'  # profile one facet; flip to None to include every facet.

shelves = fs.graph_store.list_shelves()
if FACET is not None:
    shelves = [s for s in shelves if s.facet == FACET]
by_id = {s.shelf_id: s for s in shelves}

# --- Shelf-tree structure (backend-agnostic: read off parent_shelf_id) ------
children: dict[str, list] = defaultdict(list)
roots = []
for s in shelves:
    pid = s.parent_shelf_id
    if pid is not None and pid in by_id:
        children[pid].append(s)
    else:
        roots.append(s)  # depth-0 root, or parent pruned out of this facet

# Branching factor across *internal* shelves (those with ≥1 child). Leaves
# (zero children) are reported separately so they don't swamp the histogram.
branch_counts = [len(children[s.shelf_id]) for s in shelves]
n_leaves = sum(1 for b in branch_counts if b == 0)
internal_branch = [b for b in branch_counts if b > 0]
mean_branch = float(np.mean(internal_branch)) if internal_branch else 0.0

# Depth profile per facet (or single facet) for the connectivity backbone.
depth_by_facet: dict[str, Counter] = defaultdict(Counter)
for s in shelves:
    depth_by_facet[s.facet][s.depth] += 1

print(f'Layer A backbone ({FACET or "all facets"}):')
print(f'  shelves:            {len(shelves)}')
print(f'  roots (orphans):    {len(roots)}  ({", ".join(s.label[:24] for s in roots[:4])}'
      f'{" …" if len(roots) > 4 else ""})')
print(f'  leaves:             {n_leaves}')
print(f'  internal shelves:   {len(internal_branch)}  (mean branching {mean_branch:.2f})')
print(f'  max depth:          {max((s.depth for s in shelves), default=0)}')

# --- Chunk→shelf attachment degrees from real ATTACHED_TO edges (Neo4j) -----
# Falls back to projection-time chunk_count if the backend has no driver.
attach_degree: dict[str, int] = {}
synthetic_root_share = None
driver = getattr(fs.graph_store, '_driver', None)
if driver is not None:
    facet_clause = 'WHERE s.facet = $facet' if FACET is not None else ''
    with driver.session() as session:
        rows = session.run(
            f'''
            MATCH (c:Chunk)-[:ATTACHED_TO]->(s:Shelf)
            {facet_clause}
            RETURN s.shelf_id AS sid, count(c) AS n
            ''',
            facet=FACET,
        )
        attach_degree = {r['sid']: r['n'] for r in rows}
    total_edges = sum(attach_degree.values())
    # Synthetic facet root id is `facet:<name>` per fs.attach()'s orphan fallback.
    synth_id = f'facet:{FACET}' if FACET is not None else None
    if synth_id and total_edges:
        synthetic_root_share = attach_degree.get(synth_id, 0) / total_edges
    print(f'\nattachment edges (graph): {total_edges}')
    if synthetic_root_share is not None:
        flag = '  ⚠ majority — NEL coverage likely thin' if synthetic_root_share > 0.5 else ''
        print(f'  synthetic root share:   {synthetic_root_share:.1%}{flag}')
else:
    attach_degree = {s.shelf_id: s.chunk_count for s in shelves}
    print('\n(no Neo4j driver — using projection-time chunk_count for the fan-out panel)')

# ---------------------------------------------------------------------------
fig, axes = plt.subplots(2, 2, figsize=(15, 11))

# Panel 1 — branching-factor distribution (children per internal shelf).
ax = axes[0, 0]
if internal_branch:
    bins = np.arange(0.5, max(internal_branch) + 1.5, 1)
    ax.hist(internal_branch, bins=bins, color='steelblue', edgecolor='white')
    ax.axvline(mean_branch, color='crimson', linestyle='--',
               label=f'mean = {mean_branch:.2f}')
    ax.legend()
ax.set_title(f'Branching factor\n({len(internal_branch)} internal shelves, '
             f'{n_leaves} leaves not shown)')
ax.set_xlabel('children per shelf')
ax.set_ylabel('# shelves')
ax.grid(True, alpha=0.3)

# Panel 2 — depth profile (stacked by facet if FACET is None, else single bar set).
ax = axes[0, 1]
all_depths = sorted({d for c in depth_by_facet.values() for d in c})
facets_sorted = sorted(depth_by_facet)
bottoms = np.zeros(len(all_depths))
for facet in facets_sorted:
    heights = np.array([depth_by_facet[facet].get(d, 0) for d in all_depths])
    ax.bar([str(d) for d in all_depths], heights, bottom=bottoms, label=facet)
    bottoms += heights
ax.set_title('Shelf count by projection depth')
ax.set_xlabel('depth')
ax.set_ylabel('# shelves')
if len(facets_sorted) > 1:
    ax.legend(title='facet', ncol=2)
ax.grid(True, alpha=0.3)

# Panel 3 — support-weight CCDF: P(chunk_count ≥ x), log-log. A straight-ish
# line ⇒ heavy-tailed (healthy); a cliff ⇒ one shelf hoards all the support.
ax = axes[1, 0]
weights = sorted((s.chunk_count for s in shelves if s.chunk_count > 0), reverse=True)
if weights:
    ranks = np.arange(1, len(weights) + 1)
    ccdf = ranks / len(weights)
    ax.loglog(weights, ccdf, 'o-', markersize=3, color='darkgreen')
ax.set_title('Support-weight CCDF\n(chunk_count per shelf, rank-ordered)')
ax.set_xlabel('chunk_count (log)')
ax.set_ylabel('P(X ≥ chunk_count) (log)')
ax.grid(True, which='both', alpha=0.3)

# Panel 4 — attachment fan-out: top shelves by real ATTACHED_TO degree.
ax = axes[1, 1]
ranked = sorted(attach_degree.items(), key=lambda kv: -kv[1])[:15]
if ranked:
    labels = [(by_id[sid].label[:26] if sid in by_id else sid)[:26] for sid, _ in ranked]
    vals = [n for _, n in ranked]
    colors = ['crimson' if sid.startswith('facet:') else 'slateblue'
              for sid, _ in ranked]
    y = np.arange(len(labels))
    ax.barh(y, vals, color=colors)
    ax.set_yticks(y)
    ax.set_yticklabels(labels, fontsize=8)
    ax.invert_yaxis()
src = 'graph edges' if driver is not None else 'projection chunk_count'
ax.set_title(f'Attachment fan-out — top 15 shelves\n(by {src}; '
             f'crimson = synthetic root)')
ax.set_xlabel('attached chunks')
ax.grid(True, axis='x', alpha=0.3)

fig.suptitle(f'Layer A connectivity profile — {FACET or "all facets"}', fontsize=14)
plt.tight_layout()
plt.show()

## 6e. Is Layer A ready for Layer B?

Layer B clusters chunks *within* each shelf into themes. Its hard gate is
`min_chunks_per_shelf` (default 50) — a shelf with fewer attached chunks is
skipped. So "is Layer A good enough" reduces to four questions:

1. **Structurally sound?** — `fs.audit().passed` (single root, no cycles, no
   dangling parents, attachment coverage/integrity). This is the go/no-go gate.
2. **Right-sized?** — how many shelves clear `min_chunks_per_shelf` (Layer B
   will process these), how many are too small (skipped), and is there a
   dumping-ground shelf (esp. the synthetic facet root) that means Layer A
   under-resolved? That's the cell below.
3. **Coherent?** — do a shelf's chunks actually belong to it? Read
   `fs.quality_report()` sections 1 & 5 by eye; off-topic chunks → noisy themes.
4. **Embeddings present?** — Leiden needs chunk vectors; `fs.embed()` must have
   run. The cell checks this too.

Decision: **gate on `audit.passed`, then eyeball** the readiness numbers + the
quality report. There's no single score — coherence is a judgment call.

In [ ]:
# Layer B readiness — true attached-chunk counts per shelf (what Layer B sees),
# via the backend-agnostic attachment map. Run AFTER fs.attach().
from collections import Counter

_min = fs.config.layer_b.min_chunks_per_shelf
_shelves = [s for s in fs.graph_store.list_shelves() if s.facet == 'foods']
_by_id = {s.shelf_id: s for s in _shelves}

# Invert chunk->shelves into shelf->chunk_count (real edges, not projection-time).
_attach = fs.graph_store.list_chunk_shelf_attachments()
_count: Counter = Counter()
for _cid, _sids in _attach.items():
    for _sid in _sids:
        if _sid in _by_id:
            _count[_sid] += 1

# 1. Structural gate.
_audit = fs.audit()
print(f'1. STRUCTURAL GATE  ->  audit.passed = {_audit.passed}')
if not _audit.passed:
    for c in _audit.critical_failures:
        print(f'   CRITICAL: {c.section} {c.name} (metric={c.metric}, thr={c.threshold})')
    print('   Fix criticals before Layer B.\n')

# 2. Size distribution against min_chunks_per_shelf.
_clusterable = [s for s in _shelves if _count[s.shelf_id] >= _min]
_toosmall    = [s for s in _shelves if 0 < _count[s.shelf_id] < _min]
_empty       = [s for s in _shelves if _count[s.shelf_id] == 0]
_total_chunks = sum(_count.values())
print(f'\n2. SIZE (min_chunks_per_shelf = {_min})')
print(f'   clusterable (>= {_min}): {len(_clusterable)} shelves '
      f'-> Layer B will process these')
print(f'   too small (1..{_min-1}):  {len(_toosmall)} shelves -> skipped')
print(f'   empty (0 chunks):       {len(_empty)} shelves')

# Dumping-ground check: any shelf (esp. synthetic root) holding a huge share?
print('\n   biggest shelves by attached chunks:')
for _sid, _n in _count.most_common(8):
    _share = _n / _total_chunks if _total_chunks else 0
    _root = ' (SYNTHETIC ROOT)' if _sid == 'facet:foods' else ''
    print(f'     {_by_id[_sid].label[:38]:38s} {_n:>6d}  ({_share:4.0%}){_root}')
_root_n = _count.get('facet:foods', 0)
if _total_chunks and _root_n / _total_chunks > 0.30:
    print(f'   ⚠ synthetic root holds {_root_n/_total_chunks:.0%} of chunks — Layer A '
          f'under-resolved; many chunks have no specific shelf.')

# 3. Coherence pointer.
print('\n3. COHERENCE  ->  read fs.quality_report(facet=\'foods\') sections 1 & 5')
print('   (do the sample chunks on each shelf actually belong there?)')

# 4. Embeddings present?
_sample = next((c for batch in fs.chunk_store.iter_chunks() for c in batch
                if c.shelf_ids), None)
_has_emb = _sample is not None and _sample.embedding is not None
print(f'\n4. EMBEDDINGS present on attached chunks: {_has_emb}'
      + ('' if _has_emb else '  -> run fs.embed() before Layer B'))

print(f'\nVERDICT: {len(_clusterable)} shelves ready to cluster, '
      f'{_total_chunks} chunks attached. '
      + ('Structurally GO.' if _audit.passed else 'Structural FAIL — fix first.'))

### 6f. Why is 'food product' a dumping ground?

`food product` holds ~10% of attached chunks. It's a generic FoodOn umbrella
class that *should* have been dropped by the umbrella rule (drop iff
`direct_share < 0.10` AND `lifted_share > 0.85`). It survived — this cell shows
which condition it fails, which tells us the fix:

- **High direct_share** → chunks are NEL-linked straight to the generic term
  ('food'/'food product' mentions resolve to `FOODON:00001002`). Fix is upstream
  (linker / link_blocklist), not pruning.
- **Low lifted_share** → it has real descendant support but also too many direct
  hits. Same upstream cause.
- **Below umbrella_min_count** → unlikely at this size.

Note: the 2,536 'attached' chunks include everything that *lifted* to it (chunks
whose specific term was pruned, resolving to the nearest survivor). So part of
this is descendants with no better home — separate from direct mislinking.

In [ ]:
# Diagnose 'food product' against the umbrella thresholds + see what links to it.
_la = fs.config.layer_a
_fp = next((s for s in fs.graph_store.list_shelves()
            if s.facet == 'foods' and s.label == 'food product'), None)
if _fp is None:
    print('no shelf labelled \'food product\' — maybe it was merged/renamed')
else:
    _wd = _fp.chunk_count or (_fp.support_direct + _fp.support_lifted)
    _dshare = _fp.support_direct / _wd if _wd else 0
    _lshare = _fp.support_lifted / _wd if _wd else 0
    print(f'shelf {_fp.shelf_id}  foodon={_fp.foodon_id}  depth={_fp.depth}')
    print(f'  support_direct = {_fp.support_direct}')
    print(f'  support_lifted = {_fp.support_lifted}')
    print(f'  with_descendants (chunk_count) = {_wd}')
    print(f'  direct_share = {_dshare:.3f}   (umbrella drops if < {_la.umbrella_direct_share_max})')
    print(f'  lifted_share = {_lshare:.3f}   (umbrella drops if > {_la.umbrella_lifted_share_min})')
    print(f'  umbrella_min_count = {_la.umbrella_min_count}  (rule only applies if chunk_count >= this)')
    _would_drop = (_dshare < _la.umbrella_direct_share_max
                   and _lshare > _la.umbrella_lifted_share_min
                   and _wd >= _la.umbrella_min_count)
    print(f'\n  -> umbrella rule WOULD drop it: {_would_drop}')
    if not _would_drop:
        if _dshare >= _la.umbrella_direct_share_max:
            print(f'     FAILS on direct_share: {_dshare:.0%} of support is DIRECT mentions.')
            print('     Cause: the linker maps vague mentions to the generic term.')
            print('     Fix: add (surface, FOODON:00001002) pairs to layer_a.link_blocklist,')
            print('          or lower nel confidence for generic surfaces. NOT a pruning fix.')

# What surface forms link to food product? (samples from attached chunks)
from collections import Counter
_fp_id = _fp.foodon_id if _fp else None
if _fp_id:
    _surfaces: Counter = Counter()
    _n = 0
    for _batch in fs.chunk_store.iter_chunks():
        for _c in _batch:
            if _fp_id in (_c.foodon_ids or []):
                _n += 1
                for _link in _c.entity_links:
                    if _link.ontology_id == _fp_id:
                        _surfaces[_link.mention.text.lower()] += 1
            if _n >= 2000: break
        if _n >= 2000: break
    print(f'\n  direct mentions linking to {_fp_id} (top surfaces, sampled):')
    for _surf, _cnt in _surfaces.most_common(15):
        print(f'    {_cnt:>4d}  {_surf!r}')
    if not _surfaces:
        print('    (none in sample — its support is all lifted from descendants,')
        print('     meaning it survives because lifted_share <= threshold or min_count)')

In [ ]:
# WHICH PATH feeds FOODON:00001002 support? The link_blocklist only filters
# `entity_links`; the foods-facet `foodon_ids` denorm path bypasses it. If the
# term arrives via foodon_ids, the blocklist can't drop it.
_TERM = 'FOODON:00001002'
_thr = fs.config.layer_a.min_link_confidence

# Confirm the blocklist actually loaded on the live config:
_bl = {(e.surface.lower(), e.ontology_id) for e in fs.config.layer_a.link_blocklist}
print('blocklist entries for this term:',
      sorted(s for (s, t) in _bl if t == _TERM)[:20])
print(f'min_link_confidence = {_thr}\n')

via_links_only = 0   # term in entity_links, NOT in foodon_ids
via_foodon_ids = 0   # term in foodon_ids (bypasses blocklist)
would_block    = 0   # entity_link whose (surface, term) is blocklisted
both = 0
n = 0
for _batch in fs.chunk_store.iter_chunks():
    for c in _batch:
        in_links = any(
            l.ontology_id == _TERM and l.confidence >= _thr
            for l in c.entity_links
        )
        in_denorm = _TERM in (c.foodon_ids or [])
        if in_links or in_denorm:
            n += 1
            if in_denorm:
                via_foodon_ids += 1
            if in_links and not in_denorm:
                via_links_only += 1
            if in_links and in_denorm:
                both += 1
            for l in c.entity_links:
                if l.ontology_id == _TERM and (l.mention.text.lower(), _TERM) in _bl:
                    would_block += 1
                    break
        if n >= 5000:
            break
    if n >= 5000:
        break

print(f'chunks touching {_TERM} (sampled {n}):')
print(f'  via foodon_ids denorm (BYPASSES blocklist): {via_foodon_ids}')
print(f'  via entity_links only (blocklist applies):  {via_links_only}')
print(f'  via both:                                   {both}')
print(f'  entity_links that the blocklist WOULD drop: {would_block}')
print()
if via_foodon_ids > via_links_only:
    print('=> Support comes mostly via foodon_ids -> the link_blocklist CANNOT')
    print('   fix this. Need a foodon_ids-aware filter in collect_support.')
else:
    print('=> Support comes via entity_links -> blocklist should work after a')
    print('   real re-projection. Verify build_layer_a actually re-ran.')

### 6g. Why are 18% of chunks orphaned at the synthetic root?

Dropping 'food product' sent its chunks looking for a new home. Those whose
specific FoodOn term — and *every surviving ancestor* of it — was pruned fall
through to the synthetic root `facet:foods`. This cell breaks down the orphans:

- **top orphan terms** — what FoodOn ids the root's chunks actually link to.
- **best catch-all ancestors** — pruned ancestors ranked by how many orphan
  chunks each would recover if kept as a shelf. A few high-count ancestors at
  the top => whitelist them (so those chunks get a real mid-level home).
  Whether they'd survive a lower `min_support` can't be read from the built
  graph (pruned terms carry no persisted count), so whitelisting is the
  deterministic lever.
- vs **spread thin** — if recovery scatters across many low-count ancestors,
  it's a genuine long-tail; accept it and move to Layer B.

In [ ]:
# Diagnose synthetic-root orphans: what they link to + best catch-all ancestors.
from collections import Counter

_root_id = 'facet:foods'
_shelves = fs.graph_store.list_shelves()
# A term reaches a real shelf if it (or an ancestor) survived as a shelf or was
# folded into one (see_also).
_resolvable = {s.foodon_id for s in _shelves if s.facet == 'foods' and s.foodon_id}
_resolvable |= {fid for s in _shelves if s.facet == 'foods' for fid in s.see_also}

_attach = fs.graph_store.list_chunk_shelf_attachments()
_root_chunk_ids = {cid for cid, sids in _attach.items() if _root_id in sids}
print(f'{len(_root_chunk_ids)} chunks attached to {_root_id}')

_orphan_terms: Counter = Counter()
_seen = 0
for _batch in fs.chunk_store.iter_chunks():
    for c in _batch:
        if c.chunk_id in _root_chunk_ids:
            for fid in (c.foodon_ids or []):
                if fid in fs.ontology and fid not in _resolvable:
                    _orphan_terms[fid] += 1
            _seen += 1
        if _seen >= len(_root_chunk_ids): break
    if _seen >= len(_root_chunk_ids): break

print('\ntop orphan terms (chunks -> term, label):')
for fid, cnt in _orphan_terms.most_common(20):
    print(f'  {cnt:>4d}  {fid}  {fs.ontology.id_to_label(fid)!r}')

# For each orphan term, every pruned ancestor is a candidate catch-all shelf.
# Rank ancestors by total orphan-chunks they'd recover if whitelisted.
_anc_recover: Counter = Counter()
for fid, cnt in _orphan_terms.items():
    for anc in fs.ontology.id_to_ancestors(fid):
        if anc not in _resolvable:        # ancestor was also pruned
            _anc_recover[anc] += cnt

print(f'\nbest catch-all ancestors (whitelist candidates) — min_support={fs.config.layer_a.min_support}:')
for anc, recoverable in _anc_recover.most_common(15):
    print(f'  +{recoverable:>4d} chunks  {anc}  {fs.ontology.id_to_label(anc)!r}')
print('\nFew high-count ancestors at the top => whitelist them via')
print('layer_a.facet_overrides[\'foods\'].whitelist. Spread thin => long-tail; proceed.')

In [ ]:
# WHY is each real-food orphan orphaned? Trace the prune mechanism per term,
# so we pick the lever that actually helps (min_support vs max_depth vs parent
# pruned) instead of guessing. Recompute the foods support table fresh.
from foodscholar.layer_a.propagate import collect_support

_fc = fs.config.layer_a.resolve_facet('foods')
def _chunks():
    for b in fs.chunk_store.iter_chunks():
        yield from b
_support = collect_support(_chunks(), fs.ontology,
                           min_link_confidence=_fc.min_link_confidence,
                           facet='foods', link_blocklist=_fc.link_blocklist)

_shelves = fs.graph_store.list_shelves()
_surv = {s.foodon_id for s in _shelves if s.facet=='foods' and s.foodon_id}
_surv |= {fid for s in _shelves if s.facet=='foods' for fid in s.see_also}
_min = fs.config.layer_a.min_support
_cap = fs.config.layer_a.max_depth

# Probe the real-food orphans seen in 6g.
_probe = ['FOODON:00004387','FOODON:03411043','FOODON:00004777','FOODON:00004560',
          'FOODON:03311555','FOODON:03310371']
print(f'min_support={_min}  max_depth={_cap}\n')
for fid in _probe:
    lbl = fs.ontology.id_to_label(fid)
    wd = _support.with_descendants.get(fid, 0)
    direct = _support.direct.get(fid, 0)
    # Which ancestors had enough support to survive the threshold?
    anc_surv = [(a, _support.with_descendants.get(a,0)) for a in fs.ontology.id_to_ancestors(fid)]
    anc_over = [(a,n) for a,n in anc_surv if n >= _min]
    reason = []
    if wd < _min: reason.append(f'self below min_support ({wd}<{_min})')
    else: reason.append(f'self HAS support ({wd}>={_min}) -> not a threshold victim')
    if not any(a in _surv for a in fs.ontology.id_to_ancestors(fid)):
        reason.append('NO ancestor survived as a shelf')
    print(f'{fid} {lbl!r}: direct={direct} with_desc={wd}')
    print(f'   reason: {"; ".join(reason)}')
    if anc_over[:3]:
        print(f'   ancestors over min_support (would-be parents): '
              + ', '.join(f'{fs.ontology.id_to_label(a)!r}({n})' for a,n in anc_over[:3]))
    print()
print('If self HAS support but is still orphaned -> max_depth lifted it past a')
print('surviving ancestor (lower-priority). If self is below min_support AND an')
print('ancestor is over -> lowering min_support or whitelisting that ancestor helps.')

### 6h. Which mid-level categories to whitelist?

Rare foods orphan because their only ancestors are umbrellas. Whitelisting a few
*specific* mid-level food categories gives them a home — but we must avoid the
too-generic ones ('food product', 'food material', BFO 'material entity') that
would just rebuild the dumping ground.

This cell ranks candidate parents by how many **distinct real-food orphan
terms** each would adopt, prefers the **deepest** (most specific) candidate, and
runs a greedy set-cover to find the **minimal whitelist** that re-homes the most
orphans. Copy the printed `whitelist=[...]` into the config and re-project.

In [ ]:
# Recommend mid-level whitelist by SPECIFICITY: assign each orphan to its
# DEEPEST non-generic ancestor, then count orphans per category. This avoids
# broad umbrellas (greedy-by-coverage picked 'plant food product' etc.).
from foodscholar.layer_a.propagate import collect_support
from collections import Counter

# A category is too generic to be a useful parent if it's a FoodOn
# classification AXIS (metadata, not a food kind) or a top-tier umbrella.
_GENERIC_SUBSTRINGS = (' by ', 'material', 'edible food', 'food product',
                       'claim', 'consumer group', 'characteristic',
                       'organism', 'multi-component', 'food (')
def _too_generic(fid):
    if not fid.startswith('FOODON:'):   # drops BFO:* etc.
        return True
    lbl = (fs.ontology.id_to_label(fid) or '').lower()
    # bare 'food product' / 'plant food product' / 'animal food product' tier
    if lbl.endswith('food product') and lbl.count(' ') <= 2:
        return True
    return any(sub in lbl for sub in _GENERIC_SUBSTRINGS)

_fc = fs.config.layer_a.resolve_facet('foods')
def _chunks():
    for b in fs.chunk_store.iter_chunks():
        yield from b
_support = collect_support(_chunks(), fs.ontology,
                           min_link_confidence=_fc.min_link_confidence,
                           facet='foods', link_blocklist=_fc.link_blocklist)
_min = fs.config.layer_a.min_support
_shelves = fs.graph_store.list_shelves()
_surv = {s.foodon_id for s in _shelves if s.facet=='foods' and s.foodon_id}
_surv |= {fid for s in _shelves if s.facet=='foods' for fid in s.see_also}

# Real-food orphans: own support, not generic, no surviving ancestor.
_orphans = [fid for fid in _support.direct.keys()
            if _support.with_descendants.get(fid,0) >= _min
            and not _too_generic(fid)
            and fid not in _surv
            and not any(a in _surv for a in fs.ontology.id_to_ancestors(fid))]
print(f'{len(_orphans)} real-food orphan terms\n')

# id_to_ancestors is closed & ordered nearest-first? Don't assume — rank
# candidate parents of an orphan by ontology depth (deeper = more specific).
def _depth(fid):
    return len(fs.ontology.id_to_ancestors(fid))  # proxy: more ancestors = deeper

_assigned = Counter()       # category -> # orphans whose deepest non-generic parent it is
_adopts = {}                # category -> list of example orphans
_uncovered = []
for fid in _orphans:
    cands = [a for a in fs.ontology.id_to_ancestors(fid) if not _too_generic(a)]
    if not cands:
        _uncovered.append(fid); continue
    parent = max(cands, key=_depth)      # deepest = most specific
    _assigned[parent] += 1
    _adopts.setdefault(parent, []).append(fid)

print(f'specific mid-level parents (deepest non-generic), '
      f'{len(_orphans)-len(_uncovered)}/{len(_orphans)} orphans covered:')
for cat, n in _assigned.most_common(20):
    egs = ', '.join(fs.ontology.id_to_label(o) or o for o in _adopts[cat][:3])
    print(f'  +{n:>3d}  {cat}  {fs.ontology.id_to_label(cat)!r}   e.g. {egs}')
print(f'\nlong-tail with no specific parent: {len(_uncovered)} terms')

# Suggest whitelisting categories that adopt >= 3 orphans (worth a shelf).
_picks = [(c,n) for c,n in _assigned.most_common() if n >= 3]
print('\n# whitelist categories adopting >= 3 orphans:')
print('"foods": {"whitelist": [')
for fid,_ in _picks:
    print(f'    "{fid}",  # {fs.ontology.id_to_label(fid)}')
print(']}')

## 6d. Semantic consolidation (LLM-as-judge)

Rule-based collapse only catches *structural* duplicates (single-child chains). The ~5% of duplicate shelves that share **meaning but not lexical stem** — `fermented dairy product` vs `yogurt`, `ground cattle meat` vs `beef (ground)` — survive. `fs.semantic_consolidate()` embeds each shelf (label + FoodOn synonyms), finds near-duplicate **pairs** by cosine similarity, **clusters** those pairs into connected components, then asks an LLM judge — **one call per cluster** — to decide which members merge and which stay distinct, grounded in real sample chunks pulled from the store.

Clustering means a 5-shelf cereal group becomes *one* clean decision instead of ~7 fragmented pairwise verdicts (≈5× fewer LLM calls, and transitive closure happens inside the model). Two precision controls:

- **`use_few_shot`** (default on): calibration examples in the prompt that anchor the judge on meaningful distinctions (whole vs skim milk, oil vs fat) — the cheapest fix for an over-merging model.
- **`permanent_block_list`**: pairs of FoodOn ids that must *never* merge (oil/fat, olive-oil/vegetable-oil). A vetoed pair blocks its whole cluster from merging. Grow it whenever a bad merge slips through.

`auto_merge_confidence` defaults to **0.80** (instruct models report bimodal confidence — 0.80 is their "yes"). It runs **after** `fs.attach()` and is **off by default**. Embedder is `fs.embedder` (BGE-large); judge is `fs.llm`.

Workflow: **(1)** preview candidates (no LLM) → **(2)** dry-run the judge → **(3)** persist.

In [ ]:
# Swap the mock LLM for a real judge. The notebook's from_config had no
# `llm` block, so fs.llm defaulted to the built-in mock. Groq needs
# GROQ_API_KEY in your environment.
import os
from foodscholar.config import LLMConfig, ProviderConfig
from foodscholar.llm import build_llm
os.environ['GROQ_API_KEY'] = '***REMOVED-GROQ-KEY***'
assert os.environ.get('GROQ_API_KEY'), 'set GROQ_API_KEY first'
fs.llm = build_llm(LLMConfig(
    primary=ProviderConfig(provider='groq', model='llama-3.1-8b-instant'),
))
print('judge:', fs.llm.model_id)  # must NOT contain "mock"
print(fs.llm.generate('Reply with the single word: ready'))

In [ ]:
# Threshold sweep (no LLM). Embed once, then for each threshold show how many
# candidate pairs survive AND how they cluster — the cluster sizes are what
# actually matter: too loose a threshold chains everything into one hairball.
from foodscholar.layer_a.semantic_consolidation.candidates import find_candidates
from foodscholar.layer_a.semantic_consolidation.cluster import cluster_candidates
from foodscholar.layer_a.semantic_consolidation.embed import embed_shelves

_sc = fs.config.layer_a.semantic_consolidation
_shelves = [s for s in fs.graph_store.list_shelves() if s.facet == 'foods']
_by_id = {s.shelf_id: s for s in _shelves}
_embs = embed_shelves(_shelves, fs.ontology, fs.embedder, _sc)  # ~230 vectors, once
print(f'{len(_shelves)} foods shelves, {len(_embs)} embedded  '
      f'(cluster cap = {_sc.max_cluster_size})\n')

_orig = _sc.cosine_threshold
print(f'{"thresh":>6}  {"pairs":>6}  {"clusters":>8}  {"biggest":>7}  {"shelves judged":>14}')
for thr in (0.88, 0.90, 0.91, 0.92, 0.93, 0.94, 0.95):
    _sc.cosine_threshold = thr
    cand, _ = find_candidates(_embs, _by_id, _sc)
    clusters = cluster_candidates(cand, _sc.max_cluster_size)
    biggest = max((len(c) for c in clusters), default=0)
    judged = sum(len(c) for c in clusters)
    print(f'{thr:>6.2f}  {len(cand):>6d}  {len(clusters):>8d}  {biggest:>7d}  {judged:>14d}')
_sc.cosine_threshold = _orig  # restore

# Pick the threshold where 'biggest' is small (<= cap, ideally well under) and
# 'clusters' looks like real duplicate groups, not a single blob. Then set it:
# fs.config.layer_a.semantic_consolidation.cosine_threshold = 0.93

In [ ]:
# (1) Zero-cost preview — no LLM. Shows the scaffolding filter, candidates,
# how they cluster, and what the rule filters dropped. Judge off:
fs.config.layer_a.semantic_consolidation.judge_enabled = False

# What the scaffolding filter removes (FoodOn umbrella terms: no synonyms +
# classifier suffix). These never reach the judge. If you spot a REAL food
# here, prune the offending word from cfg.classifier_suffixes (the broad ones
# are 'food'/'foods'/'ingredient').
from foodscholar.layer_a.semantic_consolidation.embed import is_scaffolding
_sc = fs.config.layer_a.semantic_consolidation
_foods = [s for s in fs.graph_store.list_shelves() if s.facet == 'foods']
_scaff = [s for s in _foods if s.foodon_id and is_scaffolding(s, fs.ontology, _sc)]
print(f'scaffolding excluded: {len(_scaff)} of {len(_foods)} foods shelves')
for s in _scaff[:20]:
    print(f'  - {s.label!r}')

art = fs.semantic_consolidate(facet='foods', dry_run=True)
print()
print(art)
print(f'\n{art.candidate_count} candidate pair(s) -> {art.cluster_count} cluster(s); '
      f'{len(art.filtered_pairs)} dropped by subtype/compound filters')
for p in art.filtered_pairs[:10]:
    print(f'  filtered {p.shelf_a} ~ {p.shelf_b} '
          f'(cos={p.cosine_similarity:.3f}): {p.filtered_reason}')

In [ ]:
# DIAGNOSTIC: send ONE real cluster to the judge and show raw + parsed output.
# IMPORTANT: force-reload the sc modules so kernel edits take effect WITHOUT a
# full restart. (A running kernel caches imports — editing the .py on disk is
# not enough; this is almost certainly why earlier results didn't change.)
import importlib, foodscholar.layer_a.semantic_consolidation as _scpkg
import foodscholar.layer_a.semantic_consolidation.prompts as _p
import foodscholar.layer_a.semantic_consolidation.embed as _e
import foodscholar.layer_a.semantic_consolidation.candidates as _c
import foodscholar.layer_a.semantic_consolidation.cluster as _cl
import foodscholar.layer_a.semantic_consolidation.judge as _j
import foodscholar.layer_a.semantic_consolidation.apply as _a
import foodscholar.llm.providers as _prov
for _m in (_p, _e, _c, _cl, _j, _a, _prov, _scpkg):
    importlib.reload(_m)
print('reloaded sc modules; prompt version =', _p.PROMPT_VERSION)
# rebuild the groq client so the reloaded GroqClient.generate_json is used:
from foodscholar.config import LLMConfig, ProviderConfig
from foodscholar.llm import build_llm
import os
if os.environ.get('GROQ_API_KEY'):
    fs.llm = build_llm(LLMConfig(primary=ProviderConfig(
        provider='groq', model='llama-3.3-70b-versatile')))
print('judge:', fs.llm.model_id)

_sc = fs.config.layer_a.semantic_consolidation
_shelves = [s for s in fs.graph_store.list_shelves() if s.facet == 'foods']
_by_id = {s.shelf_id: s for s in _shelves}
_embs = _e.embed_shelves(_shelves, fs.ontology, fs.embedder, _sc)
_cand, _ = _c.find_candidates(_embs, _by_id, _sc)
_clusters = _cl.cluster_candidates(_cand, _sc.max_cluster_size)
print(f'{len(_clusters)} clusters; sizes: {sorted((len(c) for c in _clusters), reverse=True)}')

_cluster = max(_clusters, key=len)
print(f'\njudging cluster of {len(_cluster)}:')
for sid in _cluster:
    print(f'  {sid}  {_by_id[sid].label!r}')

_prompt = _j._build_prompt(_cluster, _by_id, fs.ontology, fs.chunk_store, _sc, {})
_raw = fs.llm.generate_json(_prompt, _p.JUDGE_CLUSTER_SCHEMA,
                            max_tokens=_j._token_budget(len(_cluster)))
print('\n===== RAW MODEL JSON =====')
print(_raw)
_decision = _j._parse_cluster(_cluster, _raw, fs.llm.model_id)
print('\n===== PARSED =====')
print('merge_groups:', [(g.members, g.confidence) for g in _decision.merge_groups])
print('keep_alone  :', len(_decision.keep_alone), 'shelves')
print('\n(if RAW is {} -> groq returned empty; if it has merge_groups but parsed'
      ' is empty -> parse bug; if keep_alone=all -> model declined to merge)')

In [ ]:
# (2) Dry-run WITH the judge (groq/llama via fs.llm). No writes — read the
# proposed merge groups, the uncertain ones, and any block-list vetoes.
fs.config.layer_a.semantic_consolidation.judge_enabled = True

# Permanent block-list: pairs of FoodOn ids that must NEVER merge, by id
# (order-independent). Grow this whenever the judge proposes a bad merge you
# see below. These two were caught on this corpus:
#   cow whole milk vs cow milk      — fat-level subtype, not a duplicate
#   vitamin prep vs vitamin+mineral — category vs broader category
fs.config.layer_a.semantic_consolidation.permanent_block_list = [
    ('FOODON:00005478', 'FOODON:02020891'),   # cow whole milk / cow milk
    ('FOODON:03310620', 'FOODON:03315201'),   # vitamin prep / vitamin+mineral prep
]

art = fs.semantic_consolidate(facet='foods', dry_run=True)
_thr = fs.config.layer_a.semantic_consolidation.auto_merge_confidence

print(f'applied groups (conf >= {_thr}) — would remove {art.shelves_removed} shelves:')
for g in art.applied_groups:
    print(f'  MERGE {g.members} -> {g.canonical_name!r} '
          f'(conf={g.confidence:.2f}): {g.rationale}')

print('\nuncertain groups (below threshold — NOT applied, review by hand):')
for g in art.uncertain_groups:
    print(f'  {g.members} -> {g.canonical_name!r} (conf={g.confidence:.2f}): {g.rationale}')

if art.blocked_groups:
    print('\nblocked by permanent_block_list:')
    for g in art.blocked_groups:
        print(f'  {g.members}  vetoed pairs: {g.blocked_pairs}')

# To veto another bad merge, append its FoodOn-id pair above and re-run.

In [ ]:
# (3) Persist. Applies confirmed groups (N-way), re-upserts the shelf set,
# and re-runs fs.attach() so folded shelves' chunks re-home onto the surviving
# canonical via its see_also list. Re-run audit/quality after.
#
  #set this once you're happy with the dry-run above:
fs.config.layer_a.semantic_consolidation.enabled = True
before = len([s for s in fs.graph_store.list_shelves() if s.facet == 'foods'])
art = fs.semantic_consolidate(facet='foods', dry_run=False)
after = len([s for s in fs.graph_store.list_shelves() if s.facet == 'foods'])
print(f'foods shelves: {before} -> {after}  '
      f'({len(art.applied_groups)} groups applied, -{art.shelves_removed})')
print(fs.audit())

## 7. Build Layer B (themes)

Per-shelf theme discovery via two parallel community detections, then a greedy merge.

- **Pass 1 — Similarity** clusters chunks that *discuss the same topic in similar prose* (mutual-kNN over the embeddings we just built; edges = cosine).
- **Pass 2 — Relatedness** (SiReRAG-inspired) clusters chunks that *co-mention the same FoodOn entities* (edges weighted IDF-style over shared `ontology_id`s; `tau_strict` floors low-confidence links; `max_doc_frequency` drops ubiquitous entities).
- **Merge** pairs `(S_i, R_j)` candidates; combined Jaccard over chunks+entities `≥ dedupe_threshold` (default 0.70) collapses them into a `discovery_pass="merged"` theme. Unmerged candidates pass through as single-pass themes.

The merge step is the **stronger-evidence upgrade**, not deduplication — a merged theme is grounded in *both* topical and entity coherence. If `relatedness` contributes 0 themes, the entity graph is mis-tuned (try `min_shared_ids=1` or `max_doc_frequency=0.6`); if every theme is merged, Pass 2 isn't earning compute.

Required state: chunks have embeddings (you've just finished this) and `fs.attach()` has populated Neo4j HAS_CHUNK edges. Re-run `fs.attach()` first if Neo4j drift is suspected.


In [ ]:
# Pre-build preview — see what Pass 2 (per-shelf relatedness) looks like on
# the largest shelf, and what Pass 1 (global similarity) sees across the whole
# attached corpus. Doesn't run the full pipeline; doesn't write.

from foodscholar.layer_b.builder import (
    build_shelf_relatedness_candidates,
    build_global_similarity_candidates,
)

cfg = fs.config.layer_b

# Pass 2 preview — pick the busiest shelf, build relatedness candidates.
attachments = fs.graph_store.list_chunk_shelf_attachments()
shelf_to_chunks = {}
for chunk_id, shelf_ids in attachments.items():
    for sid in shelf_ids:
        shelf_to_chunks.setdefault(sid, []).append(chunk_id)

eligible = [
    (sid, len(cids)) for sid, cids in shelf_to_chunks.items()
    if sid != "facet:foods" and len(cids) >= cfg.min_chunks_per_shelf
]
eligible.sort(key=lambda x: -x[1])
preview_shelf, n_chunks = eligible[0]
print(f"--- Pass 2 (relatedness) preview ---")
print(f"shelf: {preview_shelf!r}  attached_chunks={n_chunks}")

chunks = fs.chunk_store.get_many(shelf_to_chunks[preview_shelf])
embedded = [c for c in chunks if c.embedding is not None]
print(f"  with embeddings: {len(embedded)}/{len(chunks)} ({len(embedded)/len(chunks):.1%})")
rel_cands = build_shelf_relatedness_candidates(chunks, cfg)
print(f"  relatedness candidates: {len(rel_cands):>3}  sizes={sorted([len(c.chunk_ids) for c in rel_cands], reverse=True)[:10]}")

# Pass 1 preview — global similarity across ALL attached chunks. Issues one ES
# kNN per chunk; on ~6k chunks expect ~30-60s. Skip if you've already run §7c.
print(f"\n--- Pass 1 (global_similarity) preview ---")
synth_root = "facet:foods"
facet_shelves = {s.shelf_id for s in fs.graph_store.list_shelves() if s.facet == "foods"}
attached_chunk_ids = sorted({
    cid for cid, sids in attachments.items()
    if any(sid in facet_shelves and sid != synth_root for sid in sids)
})
print(f"global pass would see {len(attached_chunk_ids)} attached chunks")
if len(attached_chunk_ids) > cfg.global_similarity_max_chunks:
    print(f"  (exceeds cfg.global_similarity_max_chunks={cfg.global_similarity_max_chunks}; "
          f"orchestrator would skip global pass)")
else:
    glob_cands = build_global_similarity_candidates(attached_chunk_ids, fs.chunk_store, cfg)
    print(f"  global_similarity candidates: {len(glob_cands):>3}  "
          f"sizes={sorted([len(c.chunk_ids) for c in glob_cands], reverse=True)[:10]}")


In [ ]:
# Full Layer B rollout. Run with dry_run=True first to see the artifact shape
# without writing; flip dry_run=False to persist.
#
# Cost: ~10-15 minutes wall-clock for ~100-shelf rollouts; ~$0.60 LLM cost
# at default labeling.strategy="llm".

DRY_RUN = False  # flip to False once you've inspected the dry-run output

artifact = fs.build_layer_b(facet="foods", dry_run=DRY_RUN)
print(artifact.model_dump_json(indent=2))


### 7b. Tune Layer B knobs in the notebook

Four knobs shape Pass 1 (cross-shelf similarity), the Leiden split, and the cross-pass merge:

- `similarity.knn_k` — how many neighbors each chunk sees. Higher = denser graph, larger communities. Lower = sparser graph, more (smaller) communities.
- `similarity.edge_threshold` — cosine cutoff below which neighbor edges are dropped. Higher = only tight pairs, more (smaller) communities. Lower = looser graph, fewer (larger) communities.
- `leiden.resolution` — Leiden's resolution parameter. Higher = more, smaller communities. Lower = fewer, larger ones.
- `merge.dedupe_threshold` — combined-Jaccard threshold above which a (global_similarity, relatedness) candidate pair collapses into a `merged` theme. Default 0.70 is too high in practice — Pass 2 fragments per-shelf into many small candidates so chunk-Jaccard against any one Pass-1 candidate stays in the 0.50–0.65 band. Drop to 0.55 to let natural overlaps surface. See §7g for the actual distribution from the last build.

Edit the values below, then rerun the Build cell (§7) to see the effect. The defaults baked into `LayerBConfig` are `knn_k=15, edge_threshold=0.5, leiden.resolution=1.0, merge.dedupe_threshold=0.70` — the first build at those defaults produced a single 213-chunk megacluster and zero merged themes, so the §7c sweep + §7g distribution pick tighter values automatically.

In [ ]:
# In-notebook knob overrides. Tune these, then rerun the §7 build cell.
# Comment a line out to keep the LayerBConfig default.

fs.config.layer_b.similarity.knn_k = 8                # was 15 — sparser graph
fs.config.layer_b.similarity.edge_threshold = 0.65    # was 0.50 — tighter pairs
fs.config.layer_b.leiden.resolution = 1.5             # was 1.0 — more communities
fs.config.layer_b.merge.dedupe_threshold = 0.55       # was 0.70 — let natural overlaps merge

print(f"knn_k                   = {fs.config.layer_b.similarity.knn_k}")
print(f"edge_threshold          = {fs.config.layer_b.similarity.edge_threshold}")
print(f"leiden.resolution       = {fs.config.layer_b.leiden.resolution}")
print(f"merge.dedupe_threshold  = {fs.config.layer_b.merge.dedupe_threshold}")

### 7c. Knob sweep — what values minimize megaclusters?

A real build is several minutes; a 7×3 sweep at that cost is hours. Instead, this cell:

1. Pre-fetches every attached-chunk embedding ONCE (the only ES round-trip).
2. For each knob value, re-runs `build_global_similarity_graph` + Leiden purely in-memory against the cached embeddings — ~10-30s per point.
3. Plots theme count, median theme size, largest theme size, and cross-shelf fraction vs. each knob.

The "good" region is where you avoid mega-themes (largest_size < 5% of corpus) without fragmenting into too many tiny themes. Marked dot = currently configured value; vertical dashed line = the proposed sweet spot.


In [ ]:
# Layer B Pass-1 sweep. ~2-5 minutes wall-clock on a 6k-chunk corpus.
# Embeds the chunks once (cached on the chunk store) then runs the sim+Leiden
# pipeline in-process per knob value. No ES per point, no Neo4j writes.

import time
from copy import deepcopy

import igraph as ig
import matplotlib.pyplot as plt
import numpy as np

from foodscholar.layer_b.community import run_leiden
from foodscholar.layer_b.semantic_graph import build_global_similarity_graph

# --- 1. Snapshot attached chunks once ---
FACET = "foods"
attachments = fs.graph_store.list_chunk_shelf_attachments()
facet_shelves = {s.shelf_id for s in fs.graph_store.list_shelves() if s.facet == FACET}
synth_root = f"facet:{FACET}"
attached_chunk_ids = sorted({
    cid for cid, sids in attachments.items()
    if any(sid in facet_shelves and sid != synth_root for sid in sids)
})
print(f"sweeping {len(attached_chunk_ids)} attached chunks")
print("(this prefetches all chunk embeddings in one ES round-trip per knob "
      "value via the existing chunk_store.get_many path — the global graph "
      "builder will still issue per-chunk kNN calls, but ES kNN is fast)")


# --- 2. Sweep ---
def _summarize(g, communities, n_attached):
    sizes = sorted((len(c) for c in communities), reverse=True)
    if not sizes:
        return {"n_themes": 0, "median_size": 0, "max_size": 0, "edges": g.ecount()}
    return {
        "n_themes": len(sizes),
        "median_size": int(np.median(sizes)),
        "max_size": sizes[0],
        "max_size_pct": sizes[0] / n_attached,
        "edges": g.ecount(),
    }


def _run(knn_k, edge_thr, resolution):
    sim_cfg = deepcopy(fs.config.layer_b.similarity)
    sim_cfg.knn_k = knn_k
    sim_cfg.edge_threshold = edge_thr
    leiden_cfg = deepcopy(fs.config.layer_b.leiden)
    leiden_cfg.resolution = resolution
    t0 = time.time()
    g = build_global_similarity_graph(attached_chunk_ids, fs.chunk_store, sim_cfg)
    communities = run_leiden(g, leiden_cfg)
    dt = time.time() - t0
    summary = _summarize(g, communities, len(attached_chunk_ids))
    summary["seconds"] = round(dt, 1)
    return summary


# 1-D sweeps — vary one knob at a time around the configured value
KNN_AXIS = [4, 6, 8, 10, 12, 15, 20]
THR_AXIS = [0.40, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75]
RES_AXIS = [0.5, 0.8, 1.0, 1.2, 1.5, 2.0, 2.5]

# Hold the OTHER two at the current notebook overrides
cur_knn = fs.config.layer_b.similarity.knn_k
cur_thr = fs.config.layer_b.similarity.edge_threshold
cur_res = fs.config.layer_b.leiden.resolution

print(f"\nSweep 1: knn_k @ thr={cur_thr}, res={cur_res}")
knn_results = [_run(k, cur_thr, cur_res) for k in KNN_AXIS]
for k, r in zip(KNN_AXIS, knn_results):
    print(f"  knn={k:2d} → themes={r['n_themes']:3d}  median={r['median_size']:4d}  "
          f"max={r['max_size']:4d} ({r['max_size_pct']:.0%})  edges={r['edges']:5d}  "
          f"[{r['seconds']:.1f}s]")

print(f"\nSweep 2: edge_threshold @ knn={cur_knn}, res={cur_res}")
thr_results = [_run(cur_knn, t, cur_res) for t in THR_AXIS]
for t, r in zip(THR_AXIS, thr_results):
    print(f"  thr={t:.2f} → themes={r['n_themes']:3d}  median={r['median_size']:4d}  "
          f"max={r['max_size']:4d} ({r['max_size_pct']:.0%})  edges={r['edges']:5d}  "
          f"[{r['seconds']:.1f}s]")

print(f"\nSweep 3: leiden.resolution @ knn={cur_knn}, thr={cur_thr}")
# Resolution sweep can reuse a single graph — only Leiden changes.
sim_cfg = deepcopy(fs.config.layer_b.similarity)
sim_cfg.knn_k = cur_knn
sim_cfg.edge_threshold = cur_thr
g_shared = build_global_similarity_graph(attached_chunk_ids, fs.chunk_store, sim_cfg)
res_results = []
for r_val in RES_AXIS:
    leiden_cfg = deepcopy(fs.config.layer_b.leiden)
    leiden_cfg.resolution = r_val
    t0 = time.time()
    comms = run_leiden(g_shared, leiden_cfg)
    res_results.append({**_summarize(g_shared, comms, len(attached_chunk_ids)),
                        "seconds": round(time.time() - t0, 1)})
for r_val, r in zip(RES_AXIS, res_results):
    print(f"  res={r_val:.1f} → themes={r['n_themes']:3d}  median={r['median_size']:4d}  "
          f"max={r['max_size']:4d} ({r['max_size_pct']:.0%})  [{r['seconds']:.1f}s]")


# --- 3. Plot the three sweeps in a single 3x2 grid ---
fig, axes = plt.subplots(3, 2, figsize=(12, 11))
fig.suptitle("Layer B Pass-1 knob sweeps", fontsize=14)

def _plot(ax, axis, results, key, ylabel, title, current_val):
    ax.plot(axis, [r[key] for r in results], "o-", lw=2)
    ax.axvline(current_val, color="grey", ls="--", alpha=0.6,
               label=f"current = {current_val}")
    ax.set_xlabel(title.split(" — ")[0])
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(alpha=0.3)
    ax.legend(loc="best", fontsize=8)


_plot(axes[0, 0], KNN_AXIS, knn_results, "n_themes",
      "themes", "knn_k — theme count", cur_knn)
_plot(axes[0, 1], KNN_AXIS, knn_results, "max_size",
      "largest theme size", "knn_k — max theme size", cur_knn)
_plot(axes[1, 0], THR_AXIS, thr_results, "n_themes",
      "themes", "edge_threshold — theme count", cur_thr)
_plot(axes[1, 1], THR_AXIS, thr_results, "max_size",
      "largest theme size", "edge_threshold — max theme size", cur_thr)
_plot(axes[2, 0], RES_AXIS, res_results, "n_themes",
      "themes", "leiden.resolution — theme count", cur_res)
_plot(axes[2, 1], RES_AXIS, res_results, "max_size",
      "largest theme size", "leiden.resolution — max theme size", cur_res)

# Mark "megacluster" line at 5% of corpus on the max-size plots
megaline = 0.05 * len(attached_chunk_ids)
for ax in (axes[0, 1], axes[1, 1], axes[2, 1]):
    ax.axhline(megaline, color="red", ls=":", alpha=0.6,
               label=f"5% of corpus = {int(megaline)}")
    ax.legend(loc="best", fontsize=8)

plt.tight_layout()
plt.show()

# --- 4. Recommendation ---
def _pick_sweet_spot(axis, results, n_attached):
    """Pick the largest theme-count point whose max_size is below the
    5%-of-corpus megacluster line. Falls back to the min-max-size point."""
    megaline = 0.05 * n_attached
    ok = [(a, r) for a, r in zip(axis, results) if r["max_size"] <= megaline]
    if ok:
        return max(ok, key=lambda x: x[1]["n_themes"])[0]
    return min(zip(axis, results), key=lambda x: x[1]["max_size"])[0]


reco_knn = _pick_sweet_spot(KNN_AXIS, knn_results, len(attached_chunk_ids))
reco_thr = _pick_sweet_spot(THR_AXIS, thr_results, len(attached_chunk_ids))
reco_res = _pick_sweet_spot(RES_AXIS, res_results, len(attached_chunk_ids))
print(f"\nproposed sweet spot:")
print(f"  fs.config.layer_b.similarity.knn_k           = {reco_knn}")
print(f"  fs.config.layer_b.similarity.edge_threshold  = {reco_thr}")
print(f"  fs.config.layer_b.leiden.resolution          = {reco_res}")
print("\napply with the §7b cell above, then rerun the §7 build.")


### 7d. Retrospective — theme shape after the build

After running §7 with the new knobs, this cell answers: did the tune actually reshape the themes? Three views:

- **theme size histogram** — peaked at small sizes is good; a long right tail is the megacluster signature
- **shelves-per-theme histogram** — cross-shelf themes (≥2) are the whole point of `global_similarity`; many themes pinned to a single shelf means Pass 1 didn't bridge anything
- **theme-count by pass** — sanity check on the `global_similarity` / `relatedness` / `merged` split


In [ ]:
# Retrospective post-build charts. Run AFTER §7 with the new knobs.
from collections import Counter

import matplotlib.pyplot as plt

themes = fs.graph_store.list_themes()
themes = [t for t in themes if t.facet == "foods"]

sizes = sorted([t.chunk_count for t in themes], reverse=True)
shelves_per_theme = [len(t.shelf_ids) for t in themes]
by_pass = Counter(t.discovery_pass for t in themes)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# (1) theme size histogram (log y, log-ish bins so the tail is visible)
axes[0].hist(sizes, bins=30)
axes[0].set_yscale("log")
axes[0].axvline(0.05 * sum(sizes) / max(1, len(sizes)) * 20, color="red", ls=":",
                alpha=0.6, label="5% of corpus")
axes[0].set_xlabel("chunks per theme")
axes[0].set_ylabel("themes (log)")
axes[0].set_title(f"theme size — n={len(themes)}, max={max(sizes) if sizes else 0}")
axes[0].grid(alpha=0.3)

# (2) shelves per theme
shelves_counter = Counter(shelves_per_theme)
xs = sorted(shelves_counter)
axes[1].bar(xs, [shelves_counter[x] for x in xs])
axes[1].set_xlabel("shelves attached to theme")
axes[1].set_ylabel("themes")
cross = sum(c for n, c in shelves_counter.items() if n >= 2)
axes[1].set_title(f"shelves per theme — cross-shelf = {cross}/{len(themes)}")
axes[1].grid(alpha=0.3, axis="y")

# (3) themes by pass
passes = list(by_pass)
axes[2].bar(passes, [by_pass[p] for p in passes])
axes[2].set_xlabel("discovery_pass")
axes[2].set_ylabel("themes")
axes[2].set_title("themes by pass")
axes[2].grid(alpha=0.3, axis="y")
for i, p in enumerate(passes):
    axes[2].text(i, by_pass[p], str(by_pass[p]), ha="center", va="bottom")

plt.tight_layout()
plt.show()

# Largest themes (top 10) — useful for eyeballing megaclusters
print("\nTop 10 largest themes:")
top = sorted(themes, key=lambda t: -t.chunk_count)[:10]
for t in top:
    print(f"  {t.chunk_count:4d} chunks, {len(t.shelf_ids):2d} shelves, "
          f"[{t.discovery_pass:18s}] {t.label!r}")


### 7e. Theme × shelf distribution heatmap — is Pass 1 actually bridging shelves?

§7d's marginals tell you *how many* themes are cross-shelf and *how big* they are. This cell shows you **where the chunks of each theme actually live** as a matrix: rows are themes, columns are shelves, each cell is the share of that theme's chunks attached to that shelf.

**Layout.** Rows are grouped into three strata by `discovery_pass` — `merged` on top, `global_similarity` in the middle, `relatedness` on the bottom — separated by dashed lines. The top-N themes by `chunk_count` are shown *per stratum* so all three passes get visible representation.

**Column filter.** A shelf appears as a column only if at least one shown theme has ≥ `COL_SHARE_THRESHOLD` (default 0.05) of its chunks on that shelf. Without this filter the foods facet's ~230 shelves would dominate the matrix and bury the signal.

**Color scale.** `vmax` auto-clips to the P99 cell value (floored at 0.10). Row-normalized shares are typically 0.02–0.4 even for clean themes; a fixed `vmax=1.0` compresses everything informative into the dark end of the colormap.

**Reading each row:**

| Row signature                       | Meaning                                                                                       |
|-------------------------------------|-----------------------------------------------------------------------------------------------|
| inline `█` bar + chunk count         | `log10(theme size)`; long bar + smeared row = megacluster signature                            |
| one bright cell                      | single-shelf theme — expected for `relatedness` (`r`)                                          |
| two or three bright cells            | genuine cross-shelf bridge — the point of `global_similarity` (`g`) and `merged` (`m`)         |
| long smear of dim cells              | diffuse theme — tune `knn_k` down or `edge_threshold` up (§7b/§7c)                            |

**Megacluster warning.** If > 30% of shown themes have `max share < 0.2`, the cell prints a warning. That's the build's main quality canary at this layer.

In [ ]:
# Theme × shelf distribution heatmap (foods facet). Run AFTER §7.
# Row-normalized: each cell is the fraction of that theme's chunks attached
# to that shelf. Top-N themes per discovery_pass bucket; columns trimmed to
# shelves any shown theme has non-trivial share in. Colormap auto-clipped
# at the P99 cell value so weak signal isn't compressed into one purple band.

from collections import Counter, defaultdict

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

TOP_N = 25                    # themes per discovery_pass bucket
MAX_LABEL_LEN = 40
COL_SHARE_THRESHOLD = 0.05    # keep shelves where at least one shown theme has share >= this
VMAX_PERCENTILE = 99          # color scale upper bound = this percentile of nonzero cells
PASS_ORDER = ("merged", "global_similarity", "relatedness")
PASS_INITIAL = {"merged": "m", "global_similarity": "g", "relatedness": "r"}
PASS_COLOR = {
    "merged":             "#1f77b4",
    "global_similarity":  "#2ca02c",
    "relatedness":        "#d62728",
}

# --- 1. Pull themes + shelves for the foods facet ---------------------------
all_themes = [t for t in fs.graph_store.list_themes() if t.facet == "foods"]
shelves = [s for s in fs.graph_store.list_shelves() if s.facet == "foods"]
shelf_label = {s.shelf_id: s.label for s in shelves}

if not all_themes:
    print("No foods themes found. Run §7 first.")
else:
    # --- 2. Pick top-N per bucket so all three passes are visible ---------
    by_pass = defaultdict(list)
    for t in all_themes:
        by_pass[t.discovery_pass].append(t)
    for p in by_pass:
        by_pass[p].sort(key=lambda t: -t.chunk_count)

    selected = []
    for p in PASS_ORDER:
        selected.extend(by_pass.get(p, [])[:TOP_N])

    # --- 3. Build chunk -> shelves map (one round-trip) -------------------
    chunk_shelves = fs.graph_store.list_chunk_shelf_attachments()

    # --- 4. Build raw count matrix M[theme, shelf] ------------------------
    shelf_ids_all = [s.shelf_id for s in shelves]
    shelf_idx = {sid: j for j, sid in enumerate(shelf_ids_all)}
    M = np.zeros((len(selected), len(shelf_ids_all)), dtype=np.int64)

    for i, t in enumerate(selected):
        chunk_ids = fs.graph_store.get_chunks_for_theme(t.theme_id)
        for cid in chunk_ids:
            for sid in chunk_shelves.get(cid, ()):
                j = shelf_idx.get(sid)
                if j is not None:
                    M[i, j] += 1

    # --- 5. Row-normalize FIRST (so column filter is on shares, not counts)
    row_sums = M.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1
    P_full = M / row_sums

    # --- 6. Tight column filter: keep shelves with at least one strong cell
    col_max_share = P_full.max(axis=0)
    keep_cols = np.where(col_max_share >= COL_SHARE_THRESHOLD)[0]
    if keep_cols.size == 0:
        # Fallback: all themes are extremely diffuse — show the 30 shelves
        # carrying the most mass, otherwise the chart is empty
        col_mass = M.sum(axis=0)
        keep_cols = np.argsort(-col_mass)[:30]
        keep_cols.sort()
        diffuse_fallback = True
    else:
        diffuse_fallback = False

    P = P_full[:, keep_cols]
    col_labels = [shelf_label[shelf_ids_all[j]] for j in keep_cols]
    col_labels = [c[:24] + "…" if len(c) > 25 else c for c in col_labels]

    # --- 7. Reorder columns via hierarchical clustering on shelf co-occur
    if P.shape[1] > 2:
        try:
            from scipy.cluster.hierarchy import linkage, leaves_list
            from scipy.spatial.distance import pdist
            binary = (P > 0.01).astype(np.float64).T
            d = pdist(binary, metric="jaccard")
            if np.isfinite(d).all() and d.size > 0 and d.max() > 0:
                Z = linkage(d, method="average")
                order = leaves_list(Z)
            else:
                order = np.argsort(-P.sum(axis=0))
        except ImportError:
            order = np.argsort(-P.sum(axis=0))
    else:
        order = np.arange(P.shape[1])

    P = P[:, order]
    col_labels = [col_labels[k] for k in order]

    # --- 8. Auto color scale -- clip to a high percentile of nonzero cells
    nonzero = P[P > 0]
    if nonzero.size > 0:
        vmax = float(np.percentile(nonzero, VMAX_PERCENTILE))
        vmax = max(vmax, 0.10)   # never compress useful signal under 0.10
        vmax = min(vmax, 1.0)
    else:
        vmax = 1.0

    # --- 9. Row labels with inline Unicode size bar ----------------------
    sizes = np.array([t.chunk_count for t in selected])
    log_sizes = np.log10(sizes + 1)
    bar_max = log_sizes.max() if log_sizes.size else 1.0
    BAR_WIDTH = 6
    def _bar(ls):
        filled = int(round(BAR_WIDTH * ls / bar_max))
        return "█" * filled + "·" * (BAR_WIDTH - filled)
    bars = [_bar(ls) for ls in log_sizes]

    def _fmt_label(theme, bar):
        label = (theme.label or "(unlabeled)")[:MAX_LABEL_LEN]
        return f"{bar} {theme.chunk_count:>4d}  {label}"

    row_labels_left = [_fmt_label(t, b) for t, b in zip(selected, bars)]

    # --- 10. Bucket boundaries ------------------------------------------
    bucket_starts = {}
    for i, t in enumerate(selected):
        bucket_starts.setdefault(t.discovery_pass, i)
    boundaries = sorted(bucket_starts.values())[1:]

    # --- 11. Render -----------------------------------------------------
    n_rows, n_cols = P.shape
    fig_w = max(12.0, 0.50 * n_cols + 6.0)
    fig_h = max(6.0, 0.34 * n_rows + 2.5)
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))

    im = ax.imshow(P, aspect="auto", cmap="magma", vmin=0.0, vmax=vmax,
                   interpolation="nearest")

    ax.set_xticks(np.arange(n_cols))
    ax.set_xticklabels(col_labels, rotation=45, ha="left", fontsize=9)
    ax.xaxis.tick_top()
    ax.xaxis.set_label_position("top")

    ax.set_yticks(np.arange(n_rows))
    ax.set_yticklabels(row_labels_left, fontsize=9, family="monospace")

    ax2 = ax.twinx()
    ax2.set_ylim(ax.get_ylim())
    ax2.set_yticks(np.arange(n_rows))
    right_labels = [PASS_INITIAL[t.discovery_pass] for t in selected]
    ax2.set_yticklabels(right_labels, fontsize=10, family="monospace", fontweight="bold")
    for tick_label, theme in zip(ax2.get_yticklabels(), selected):
        tick_label.set_color(PASS_COLOR[theme.discovery_pass])

    for b in boundaries:
        ax.axhline(b - 0.5, color="white", linewidth=2.0)
        ax.axhline(b - 0.5, color="black", linewidth=0.6, linestyle="--")

    bucket_counts = Counter(t.discovery_pass for t in selected)
    title = (
        f"theme × shelf distribution  —  foods facet  "
        f"(shown: m={bucket_counts.get('merged', 0)} "
        f"g={bucket_counts.get('global_similarity', 0)} "
        f"r={bucket_counts.get('relatedness', 0)}; "
        f"{n_cols} shelves; color clipped at P{VMAX_PERCENTILE}={vmax:.2f})"
    )
    ax.set_title(title, fontsize=11, pad=12)

    cbar = fig.colorbar(im, ax=ax2, shrink=0.7, pad=0.04)
    cbar.set_label(f"share of theme's chunks (clip at {vmax:.2f})", fontsize=9)
    cbar.ax.tick_params(labelsize=8)

    legend_handles = [
        mpatches.Patch(color=PASS_COLOR[p], label=f"{PASS_INITIAL[p]} = {p}")
        for p in PASS_ORDER if p in bucket_counts
    ]
    ax.legend(handles=legend_handles, loc="upper left", bbox_to_anchor=(0.0, -0.04),
              ncol=3, frameon=False, fontsize=9)

    ax.set_yticks(np.arange(-0.5, n_rows, 1), minor=True)
    ax.grid(which="minor", color="#e0e0e0", linewidth=0.4)
    ax.tick_params(which="minor", length=0)

    plt.tight_layout()
    plt.show()

    # --- 12. Summary (now with diffuse-warning) -------------------------
    max_per_row = P_full.max(axis=1)
    n_single = sum(1 for t in selected if len(t.shelf_ids) == 1)
    n_bridge = sum(1 for t in selected if len(t.shelf_ids) >= 2)
    n_diffuse = int(np.sum(max_per_row < 0.5))
    n_severely_diffuse = int(np.sum(max_per_row < 0.2))

    print(
        f"shown: single-shelf={n_single}  bridge(>=2 shelves)={n_bridge}  "
        f"diffuse(max share<0.5)={n_diffuse}  "
        f"severely diffuse(<0.2)={n_severely_diffuse}"
    )
    print(f"by pass (shown):  {dict(bucket_counts)}")
    print(f"by pass (total):  {dict(Counter(t.discovery_pass for t in all_themes))}")
    print(f"shelves shown / shelves on facet:  {n_cols} / {len(shelves)}")
    if diffuse_fallback:
        print(
            "⚠ no shelf hit the COL_SHARE_THRESHOLD floor — every theme is spread thin. "
            "Fell back to top-30 shelves by mass. Re-tune §7b/§7c (knn_k down or "
            "edge_threshold up); the megacluster signature is visible in `severely diffuse`."
        )
    elif n_severely_diffuse > 0.3 * n_rows:
        print(
            f"⚠ {n_severely_diffuse}/{n_rows} themes have max share < 0.2 — "
            "the build is producing megaclusters. Re-tune §7b/§7c before trusting Layer C."
        )


### 7f. Per-theme embedding coherence — are the clusters tight?

§7e tells you *where* a theme's chunks land across shelves. This cell tells you whether the chunks themselves are tight in embedding space — i.e. whether the label has a real semantic focus to describe, or whether the cluster is a topological artifact (Leiden grouped them but they don't share content).

**Metric.** For each theme, compute the **mean pairwise cosine** of its chunk embeddings. Embeddings are L2-normalized, so cosine = dot product. For themes with > `MAX_CHUNKS_PER_THEME` chunks, sample down for tractability (variance is small at n≥80).

**Reading the histogram.**

| Mean-cosine range | Interpretation                                                    |
|-------------------|-------------------------------------------------------------------|
| ≥ 0.70             | tight, focused theme — labels will work                            |
| 0.55 – 0.70        | acceptable — typical for a mature build                            |
| 0.40 – 0.55        | loose — label will be vague, c-TF-IDF will pick generic terms      |
| < 0.40             | incoherent — Leiden grouped these but they don't share content    |

Faceted by `discovery_pass` because each pass has a different shape: `merged` should peak highest (both signals agreed), `relatedness` can be lower-coherence-but-still-valid (entity vocabulary, not prose), `global_similarity` sits in between. If `relatedness` peaks higher than `global_similarity`, Pass 1 is over-clustering.

**The labeled-but-weak failure mode.** A theme can pass §7d (size OK) and §7e (one bright shelf cell) yet still have mean coherence < 0.4 — the chunks are nominally on one shelf but discuss unrelated sub-topics. The c-TF-IDF label will name one of the sub-topics and look fine; only this metric catches the mismatch.

In [ ]:
# Per-theme embedding coherence — mean pairwise cosine on each theme's chunks.
# Embeddings are already L2-normalized in ES (cosine HNSW), so pairwise cosine
# = pairwise dot product. Run AFTER §7. ~1-3 minutes on a ~150-theme build.

import random
from collections import defaultdict

import matplotlib.pyplot as plt
import numpy as np

MAX_CHUNKS_PER_THEME = 80   # downsample for speed; variance is small past this
MIN_CHUNKS_PER_THEME = 4    # skip pathologically small themes
RNG_SEED = 42

PASS_ORDER = ("merged", "global_similarity", "relatedness")
PASS_COLOR = {
    "merged":             "#1f77b4",
    "global_similarity":  "#2ca02c",
    "relatedness":        "#d62728",
}

rng = random.Random(RNG_SEED)
themes = [t for t in fs.graph_store.list_themes() if t.facet == "foods"]

if not themes:
    print("No foods themes found. Run §7 first.")
else:
    # --- 1. For each theme, sample chunk_ids and fetch embeddings -----------
    print(f"computing coherence for {len(themes)} themes...")
    coherences: dict[str, float] = {}
    sizes_used: dict[str, int] = {}

    # Cache: a chunk may belong to multiple themes; pull each embedding once.
    embedding_cache: dict[str, np.ndarray] = {}

    def _fetch_embeddings(chunk_ids: list[str]) -> list[np.ndarray]:
        missing = [c for c in chunk_ids if c not in embedding_cache]
        if missing:
            for chunk in fs.chunk_store.get_many(missing):
                if chunk.embedding is not None:
                    embedding_cache[chunk.chunk_id] = np.asarray(chunk.embedding, dtype=np.float32)
        return [embedding_cache[c] for c in chunk_ids if c in embedding_cache]

    skipped_no_emb = 0
    skipped_small = 0
    for t in themes:
        chunk_ids = fs.graph_store.get_chunks_for_theme(t.theme_id)
        if len(chunk_ids) < MIN_CHUNKS_PER_THEME:
            skipped_small += 1
            continue
        if len(chunk_ids) > MAX_CHUNKS_PER_THEME:
            chunk_ids = rng.sample(chunk_ids, MAX_CHUNKS_PER_THEME)

        vecs = _fetch_embeddings(chunk_ids)
        if len(vecs) < MIN_CHUNKS_PER_THEME:
            skipped_no_emb += 1
            continue

        E = np.stack(vecs)
        # Re-normalize defensively — ES round-trip can leave tiny drift.
        norms = np.linalg.norm(E, axis=1, keepdims=True)
        norms[norms == 0] = 1.0
        E = E / norms

        # Pairwise cosine = E @ E.T; mean of upper triangle excluding diagonal
        sim = E @ E.T
        n = sim.shape[0]
        triu_idx = np.triu_indices(n, k=1)
        coherences[t.theme_id] = float(sim[triu_idx].mean())
        sizes_used[t.theme_id] = n

    n_eval = len(coherences)
    print(f"  evaluated: {n_eval}  skipped(too small): {skipped_small}  "
          f"skipped(no embedding): {skipped_no_emb}")

    if n_eval == 0:
        print("Nothing to plot — every theme was too small or missing embeddings.")
    else:
        # --- 2. Group coherence values by discovery_pass -------------------
        by_pass: dict[str, list[float]] = defaultdict(list)
        theme_by_id = {t.theme_id: t for t in themes}
        for tid, c in coherences.items():
            by_pass[theme_by_id[tid].discovery_pass].append(c)

        # --- 3. Plot: stacked histogram + overlaid pass-faceted KDE-ish ----
        bins = np.linspace(0.0, 1.0, 41)

        fig, (ax_hist, ax_facet) = plt.subplots(1, 2, figsize=(13, 4.2))

        # Left: stacked histogram (one bar per bin, color-stacked by pass)
        stack_data = [by_pass.get(p, []) for p in PASS_ORDER]
        stack_colors = [PASS_COLOR[p] for p in PASS_ORDER]
        stack_labels = [
            f"{p} (n={len(by_pass.get(p, []))})" for p in PASS_ORDER
        ]
        ax_hist.hist(stack_data, bins=bins, stacked=True, color=stack_colors,
                     label=stack_labels, edgecolor="white", linewidth=0.4)
        for x, txt in [(0.40, "incoherent"), (0.55, "loose"), (0.70, "tight")]:
            ax_hist.axvline(x, color="#444", linestyle=":", linewidth=0.8)
            ax_hist.text(x, ax_hist.get_ylim()[1] * 0.92 if False else 0.5,
                         txt, rotation=90, va="bottom", ha="right",
                         fontsize=8, color="#555")
        ax_hist.set_xlabel("mean pairwise cosine (theme coherence)")
        ax_hist.set_ylabel("theme count")
        ax_hist.set_title(f"coherence distribution (n={n_eval} themes)")
        ax_hist.legend(loc="upper left", fontsize=8, frameon=False)
        ax_hist.set_xlim(0.0, 1.0)

        # Right: per-pass overlay (step histograms, normalized)
        for p in PASS_ORDER:
            vals = by_pass.get(p, [])
            if not vals:
                continue
            ax_facet.hist(vals, bins=bins, histtype="step", linewidth=2.0,
                          color=PASS_COLOR[p], label=f"{p} (n={len(vals)})",
                          density=True)
        for x in (0.40, 0.55, 0.70):
            ax_facet.axvline(x, color="#888", linestyle=":", linewidth=0.6)
        ax_facet.set_xlabel("mean pairwise cosine")
        ax_facet.set_ylabel("density")
        ax_facet.set_title("per-pass overlay (normalized)")
        ax_facet.legend(loc="upper left", fontsize=8, frameon=False)
        ax_facet.set_xlim(0.0, 1.0)

        plt.tight_layout()
        plt.show()

        # --- 4. Summary stats + per-pass medians ---------------------------
        all_vals = np.array(list(coherences.values()))
        print(f"\nall themes: median={np.median(all_vals):.3f}  "
              f"P25={np.percentile(all_vals,25):.3f}  "
              f"P75={np.percentile(all_vals,75):.3f}")
        for p in PASS_ORDER:
            vals = np.array(by_pass.get(p, []))
            if vals.size:
                print(f"  {p:18s}: n={vals.size:3d}  median={np.median(vals):.3f}  "
                      f"P25={np.percentile(vals,25):.3f}  P75={np.percentile(vals,75):.3f}")

        # Buckets and warnings
        n_tight   = int(np.sum(all_vals >= 0.70))
        n_ok      = int(np.sum((all_vals >= 0.55) & (all_vals < 0.70)))
        n_loose   = int(np.sum((all_vals >= 0.40) & (all_vals < 0.55)))
        n_bad     = int(np.sum(all_vals < 0.40))
        print(f"\nbucket counts:  tight(>=0.70)={n_tight}  ok=[0.55,0.70)={n_ok}  "
              f"loose=[0.40,0.55)={n_loose}  incoherent(<0.40)={n_bad}")

        # --- 5. Bottom-10 incoherent themes — the actionable list ---------
        worst = sorted(coherences.items(), key=lambda kv: kv[1])[:10]
        print("\nbottom 10 by coherence (label these for inspection or kill them):")
        for tid, c in worst:
            t = theme_by_id[tid]
            label = (t.label or "(unlabeled)")[:60]
            print(f"  {c:.3f}  [{t.discovery_pass[:5]:5s}]  size={t.chunk_count:>4d}  "
                  f"shelves={len(t.shelf_ids):>2d}  {label!r}")

        if n_bad > 0.20 * n_eval:
            print(
                f"\n⚠ {n_bad}/{n_eval} themes have coherence < 0.40 — labels will be "
                "unreliable. Re-tune §7b/§7c (higher edge_threshold or lower knn_k "
                "produces tighter clusters)."
            )
        if by_pass.get("relatedness") and by_pass.get("global_similarity"):
            rel_med = float(np.median(by_pass["relatedness"]))
            sim_med = float(np.median(by_pass["global_similarity"]))
            if rel_med > sim_med + 0.05:
                print(
                    f"⚠ relatedness median ({rel_med:.2f}) > global_similarity "
                    f"median ({sim_med:.2f}) — Pass 1 is producing looser clusters "
                    "than Pass 2. Consider raising edge_threshold."
                )


### 7g. Megablob inspector + merge-pair Jaccard distribution

Two diagnostics in one cell. Both target the issues §7e/§7f surfaced.

**Part 1 — megablob inspector.** For every theme touching ≥ `MIN_SHELVES` (default 10), print:
- top FoodOn ids by frequency across its chunks (what the chunks are about, in entity terms)
- 3 random chunk excerpts (what the chunks are about, in prose)

Lets you SEE whether a 134-shelf theme like `selecting ripe fruit` is genuinely about produce (acceptable broad theme), or a grab-bag the label happened to land on (bad theme that should be split or killed).

**Part 2 — merge-pair Jaccard distribution.** For every (global_similarity, relatedness) **theme pair** in the persisted build, compute the chunk-Jaccard between their chunk sets. Since the last build merged zero candidates (`merged_rate=0`), the persisted themes ARE the candidates — so this is the same number `MergeDecision.chunk_jaccard` would have shown.

The histogram tells you whether `dedupe_threshold=0.70` was actually the cliff:
- If a cluster of pairs sits at 0.55–0.68, the threshold was the problem. Drop it (already done in config.local.yaml).
- If everything's < 0.20, the two passes find genuinely disjoint chunks. The dual-pass architecture is duplicating effort, not complementing.
- Combined-similarity (chunk_weight · J_chunk + entity_weight · J_entity) is also plotted — that's the actual quantity compared to `dedupe_threshold`.

In [ ]:
# §7g diagnostic cell — megablob inspector + merge-pair Jaccard histogram.
# Run AFTER §7. Read-only: doesn't write back to any store.

import random
from collections import Counter, defaultdict

import matplotlib.pyplot as plt
import numpy as np

# Tunables
MIN_SHELVES_FOR_MEGABLOB = 10     # show themes touching at least this many shelves
N_MEGABLOBS_TO_SHOW = 5           # limit to top-N largest (by chunk_count)
N_CHUNK_SAMPLES = 3               # random chunk excerpts per megablob
N_TOP_FOODON = 8                  # FoodOn ids per megablob
EXCERPT_CHARS = 220
RNG_SEED = 42

# Recover the merge weights from config so the recomputed combined-similarity
# uses the same formula the next build will.
_mcfg = fs.config.layer_b.merge
CHUNK_WEIGHT = _mcfg.chunk_weight
ENTITY_WEIGHT = _mcfg.entity_weight
DEDUPE_THRESHOLD = _mcfg.dedupe_threshold

rng = random.Random(RNG_SEED)
themes = [t for t in fs.graph_store.list_themes() if t.facet == "foods"]
theme_by_id = {t.theme_id: t for t in themes}

if not themes:
    print("No foods themes found. Run §7 first.")
else:
    # =========================================================================
    # Part 1 — Megablob inspector
    # =========================================================================
    print("=" * 78)
    print("PART 1 — Megablob inspector  "
          f"(themes touching >= {MIN_SHELVES_FOR_MEGABLOB} shelves)")
    print("=" * 78)

    megablobs = [t for t in themes if len(t.shelf_ids) >= MIN_SHELVES_FOR_MEGABLOB]
    megablobs.sort(key=lambda t: -t.chunk_count)
    print(f"\nfound {len(megablobs)} megablob(s); showing top {N_MEGABLOBS_TO_SHOW}\n")

    for t in megablobs[:N_MEGABLOBS_TO_SHOW]:
        chunk_ids = fs.graph_store.get_chunks_for_theme(t.theme_id)
        chunks = fs.chunk_store.get_many(chunk_ids)

        print(f"--- [{t.discovery_pass}]  {t.label!r}")
        print(f"    size={t.chunk_count}  shelves={len(t.shelf_ids)}  "
              f"theme_id={t.theme_id}")

        # Top FoodOn ids across the theme's chunks
        foodon_counter: Counter = Counter()
        for c in chunks:
            for fid in (c.foodon_ids or []):
                foodon_counter[fid] += 1
        # Resolve to labels via fs.entities so they read
        top_foodon = foodon_counter.most_common(N_TOP_FOODON)
        if top_foodon:
            entity_index = {}
            for fid, _ in top_foodon:
                ent = fs.entities.get(fid)
                if ent is not None:
                    entity_index[fid] = ent.label or fid
                else:
                    entity_index[fid] = fid
            print(f"    top foodon ids ({sum(foodon_counter.values())} mentions, "
                  f"{len(foodon_counter)} distinct):")
            for fid, n in top_foodon:
                share = n / t.chunk_count
                print(f"      {n:>3d}  ({share:.0%})  {entity_index[fid]}  [{fid}]")

        # Sampled chunk excerpts
        sample = rng.sample(chunks, min(N_CHUNK_SAMPLES, len(chunks)))
        print(f"    sample chunks ({N_CHUNK_SAMPLES} of {len(chunks)}):")
        for i, c in enumerate(sample, 1):
            text = (c.text or "").strip().replace("\n", " ")
            if len(text) > EXCERPT_CHARS:
                text = text[:EXCERPT_CHARS] + "…"
            src = c.source_type if hasattr(c, "source_type") else "?"
            print(f"      {i}. [{src}]  {text}")
        print()

    # =========================================================================
    # Part 2 — Merge-pair Jaccard distribution
    # =========================================================================
    print("=" * 78)
    print("PART 2 — Merge-pair Jaccard distribution")
    print("=" * 78)

    # Pull chunk_ids per theme once. For relatedness themes this is one shelf;
    # for global_similarity themes it spans shelves. Either way, the set of
    # chunk_ids is what the candidate-Jaccard saw before persistence.
    g_themes = [t for t in themes if t.discovery_pass == "global_similarity"]
    r_themes = [t for t in themes if t.discovery_pass == "relatedness"]

    if not g_themes or not r_themes:
        print(
            f"\nNot enough cross-pass material: g={len(g_themes)} r={len(r_themes)}. "
            "Histogram only meaningful when both passes produced themes."
        )
    else:
        print(f"\ng_themes (global_similarity): {len(g_themes)}")
        print(f"r_themes (relatedness):        {len(r_themes)}")
        print(f"pairs to score:                {len(g_themes) * len(r_themes)}")

        # Pre-fetch chunk sets and foodon-id sets per theme (one ES round-trip
        # per theme via get_many, cached).
        chunk_sets: dict[str, set[str]] = {}
        foodon_sets: dict[str, set[str]] = {}
        for t in g_themes + r_themes:
            ids = fs.graph_store.get_chunks_for_theme(t.theme_id)
            chunk_sets[t.theme_id] = set(ids)
            foodon_acc: set[str] = set()
            for c in fs.chunk_store.get_many(ids):
                foodon_acc.update(c.foodon_ids or [])
            foodon_sets[t.theme_id] = foodon_acc

        # Compute pairwise Jaccard
        chunk_j = np.zeros((len(g_themes), len(r_themes)), dtype=np.float32)
        entity_j = np.zeros_like(chunk_j)
        for i, gt in enumerate(g_themes):
            gc, gf = chunk_sets[gt.theme_id], foodon_sets[gt.theme_id]
            for j, rt in enumerate(r_themes):
                rc, rf = chunk_sets[rt.theme_id], foodon_sets[rt.theme_id]
                u_c = len(gc | rc)
                u_f = len(gf | rf)
                chunk_j[i, j] = (len(gc & rc) / u_c) if u_c else 0.0
                entity_j[i, j] = (len(gf & rf) / u_f) if u_f else 0.0

        combined = CHUNK_WEIGHT * chunk_j + ENTITY_WEIGHT * entity_j

        # --- Plot: three histograms (chunk-J, entity-J, combined) -----------
        fig, axes = plt.subplots(1, 3, figsize=(15, 4))
        bins = np.linspace(0.0, 1.0, 41)

        for ax, data, title in zip(
            axes,
            [chunk_j.ravel(), entity_j.ravel(), combined.ravel()],
            ["chunk Jaccard", "entity Jaccard",
             f"combined  ({CHUNK_WEIGHT}·J_c + {ENTITY_WEIGHT}·J_e)"],
        ):
            ax.hist(data, bins=bins, color="#1f77b4", edgecolor="white", linewidth=0.4)
            ax.set_yscale("log")
            ax.set_xlim(0.0, 1.0)
            ax.set_xlabel(title)
            ax.set_ylabel("pair count (log)")
            ax.axvline(DEDUPE_THRESHOLD, color="red", linestyle="--",
                       linewidth=1.2, label=f"dedupe={DEDUPE_THRESHOLD}")
            ax.legend(loc="upper right", fontsize=9, frameon=False)

        fig.suptitle(
            f"merge-pair Jaccard distribution  —  {len(g_themes)}×{len(r_themes)} "
            f"= {len(g_themes) * len(r_themes)} pairs",
            fontsize=11,
        )
        plt.tight_layout()
        plt.show()

        # --- Summary stats --------------------------------------------------
        flat_c = chunk_j.ravel()
        flat_e = entity_j.ravel()
        flat_combined = combined.ravel()

        print("\nchunk Jaccard:    "
              f"max={flat_c.max():.3f}  P99={np.percentile(flat_c, 99):.3f}  "
              f"P95={np.percentile(flat_c, 95):.3f}  median={np.median(flat_c):.3f}")
        print("entity Jaccard:   "
              f"max={flat_e.max():.3f}  P99={np.percentile(flat_e, 99):.3f}  "
              f"P95={np.percentile(flat_e, 95):.3f}  median={np.median(flat_e):.3f}")
        print("combined:         "
              f"max={flat_combined.max():.3f}  P99={np.percentile(flat_combined, 99):.3f}  "
              f"P95={np.percentile(flat_combined, 95):.3f}  "
              f"median={np.median(flat_combined):.3f}")

        # Threshold sensitivity table
        print("\nhow many pairs would merge at each threshold:")
        for thresh in [0.70, 0.65, 0.60, 0.55, 0.50, 0.45, 0.40]:
            n_merged = int(np.sum(flat_combined >= thresh))
            print(f"  threshold {thresh:.2f} -> {n_merged:>5d} pairs merge "
                  f"({100.0 * n_merged / flat_combined.size:.2f}% of all pairs)")

        # Top 10 closest pairs (highest combined)
        flat_idx = np.argsort(-flat_combined)[:10]
        print("\ntop 10 closest pairs (highest combined):")
        for idx in flat_idx:
            i, j = divmod(int(idx), combined.shape[1])
            gt, rt = g_themes[i], r_themes[j]
            print(f"  combined={combined[i, j]:.3f}  "
                  f"J_c={chunk_j[i, j]:.3f}  J_e={entity_j[i, j]:.3f}  |  "
                  f"g: {gt.label!r} (n={gt.chunk_count})  ↔  "
                  f"r: {rt.label!r} (n={rt.chunk_count})")

        # Verdict
        if flat_combined.max() < DEDUPE_THRESHOLD:
            print(
                f"\n⚠ Even the closest pair has combined={flat_combined.max():.3f} "
                f"< {DEDUPE_THRESHOLD:.2f}. The dual-pass architecture is finding "
                "genuinely disjoint candidate sets — either lower dedupe_threshold "
                "further or accept that this build won't produce merged themes."
            )
        elif int(np.sum(flat_combined >= DEDUPE_THRESHOLD)) < 3:
            print(
                f"\n⚠ Only {int(np.sum(flat_combined >= DEDUPE_THRESHOLD))} pairs "
                f"are above the new dedupe_threshold ({DEDUPE_THRESHOLD}). Consider "
                "dropping further to 0.50 or 0.45 if you want a meaningful merged bucket."
            )
        else:
            print(
                f"\n✓ {int(np.sum(flat_combined >= DEDUPE_THRESHOLD))} pairs would "
                f"merge at the new dedupe_threshold ({DEDUPE_THRESHOLD}). "
                "Re-run §7 to see merged themes."
            )


### 7h. Dissecting the closest "should-have-merged" pairs

§7g revealed something structural: the highest combined-Jaccard pair (`fatty fish and seafood options` ↔ `health benefits of fish and seafood`) hits 0.356 — even though semantically they describe the same theme. Pass 1 and Pass 2 are partitioning the *same conceptual region* of the corpus into largely **disjoint chunk subsets**.

Before deciding how to fix the merge step, we need to know why. There are several mechanically distinct possibilities:

1. **Scope mismatch.** Pass 2 only sees chunks attached to a single shelf at a time. If the chunks of a theme are spread across N shelves, Pass 2 builds N separate per-shelf graphs and produces N small themes. Pass 1 pools all N shelves' chunks into one community.
2. **Edge-criterion mismatch.** Pass 1 connects chunks by BGE embedding cosine. Pass 2 connects them by shared rare-FoodOn entities. A chunk can be a strong Pass-1 neighbor (similar prose) but a weak Pass-2 neighbor (no shared entities).
3. **Resolution / min_community_size mismatch.** Both passes use the same Leiden knobs, but Pass 2 operates on smaller graphs and may fragment into many `min_community_size=15`-floored mini-communities while Pass 1 produces fewer larger ones.
4. **A chunk attribution bug.** Pass 2's `min_chunks_per_shelf=50` gate might be excluding chunks from per-shelf clustering even though Pass 1 includes them.

This cell takes the top-N closest pairs and dissects each one:
- count chunks in g-only, r-only, g∩r
- show what shelves the g-only chunks attach to (does Pass 2 even see them?)
- show 3 sampled g-only and 3 r-only chunk excerpts (are they about the same topic?)

The answer tells you whether to change merge semantics, change Pass 2's scope, or accept the multi-resolution catalog as-is.

In [ ]:
# §7h — Dissect the top-N "should-have-merged" pairs from §7g.
# Read-only. Re-uses the chunk_sets/foodon_sets logic from §7g internally.

import random
from collections import Counter

# Tunables
N_PAIRS_TO_DISSECT = 3
N_SAMPLE_EXCERPTS = 3
EXCERPT_CHARS = 200
RNG_SEED = 42

rng = random.Random(RNG_SEED)
themes = [t for t in fs.graph_store.list_themes() if t.facet == "foods"]
theme_by_id = {t.theme_id: t for t in themes}

g_themes = [t for t in themes if t.discovery_pass == "global_similarity"]
r_themes = [t for t in themes if t.discovery_pass == "relatedness"]
g_by_id = {t.theme_id: t for t in g_themes}
r_by_id = {t.theme_id: t for t in r_themes}

# Pre-fetch chunk sets per theme (one round-trip per theme, then cached).
chunk_sets: dict[str, set[str]] = {}
foodon_sets: dict[str, set[str]] = {}
for t in g_themes + r_themes:
    ids = fs.graph_store.get_chunks_for_theme(t.theme_id)
    chunk_sets[t.theme_id] = set(ids)

# Shelf attachments (one round-trip)
chunk_shelves = fs.graph_store.list_chunk_shelf_attachments()
shelves = [s for s in fs.graph_store.list_shelves() if s.facet == "foods"]
shelf_label = {s.shelf_id: s.label for s in shelves}

# Pass 2's per-shelf gate — chunks on shelves below this many attachments
# never enter the relatedness pass at all.
min_chunks_per_shelf = fs.config.layer_b.min_chunks_per_shelf
shelf_attach_count: Counter = Counter()
for cid, sids in chunk_shelves.items():
    for sid in sids:
        shelf_attach_count[sid] += 1
gated_shelves = {sid for sid, n in shelf_attach_count.items() if n < min_chunks_per_shelf}

# Re-compute combined-Jaccard for ranking (small cost since chunk_sets is cached)
_mcfg = fs.config.layer_b.merge
CHUNK_WEIGHT = _mcfg.chunk_weight
ENTITY_WEIGHT = _mcfg.entity_weight

# Use the chunk-Jaccard alone for ranking — entity-Jaccard isn't needed to
# pick the dissection targets and avoids re-fetching every chunk.
def _chunk_j(g, r):
    gc, rc = chunk_sets[g.theme_id], chunk_sets[r.theme_id]
    u = len(gc | rc)
    return (len(gc & rc) / u) if u else 0.0

ranked = []
for g in g_themes:
    for r in r_themes:
        j = _chunk_j(g, r)
        ranked.append((j, g, r))
ranked.sort(key=lambda t: -t[0])

print("=" * 90)
print(f"Dissecting top {N_PAIRS_TO_DISSECT} pairs by chunk-Jaccard")
print(f"(Pass 2 per-shelf gate: min_chunks_per_shelf = {min_chunks_per_shelf})")
print(f"(Shelves below the gate, excluded from Pass 2: {len(gated_shelves)} of {len(shelves)})")
print("=" * 90)

for rank, (j, g, r) in enumerate(ranked[:N_PAIRS_TO_DISSECT], 1):
    gc = chunk_sets[g.theme_id]
    rc = chunk_sets[r.theme_id]
    inter = gc & rc
    g_only = gc - rc
    r_only = rc - gc

    print(f"\n{'─' * 90}")
    print(f"#{rank}  J_chunk = {j:.3f}")
    print(f"  g: {g.label!r}  (size={len(gc)}, shelves={len(g.shelf_ids)})")
    print(f"  r: {r.label!r}  (size={len(rc)}, shelf={r.shelf_ids[0] if r.shelf_ids else '?'})")
    print(f"  shared:  {len(inter)}   g-only: {len(g_only)}   r-only: {len(r_only)}")

    # --- Pass 2 scope check: where do g-only chunks live? -----------------
    g_only_shelf_counter: Counter = Counter()
    g_only_on_gated: int = 0
    g_only_on_r_shelf: int = 0
    r_shelf_id = r.shelf_ids[0] if r.shelf_ids else None
    for cid in g_only:
        cshelves = chunk_shelves.get(cid, set())
        for sid in cshelves:
            g_only_shelf_counter[sid] += 1
        if cshelves & gated_shelves:
            g_only_on_gated += 1
        if r_shelf_id and r_shelf_id in cshelves:
            g_only_on_r_shelf += 1

    print(f"\n  g-only chunks ({len(g_only)}) attach to {len(g_only_shelf_counter)} distinct shelves:")
    for sid, n in g_only_shelf_counter.most_common(8):
        gate_marker = " (Pass-2 gated)" if sid in gated_shelves else ""
        rshelf_marker = " ← r's shelf" if sid == r_shelf_id else ""
        print(f"    {n:>3d}  {shelf_label.get(sid, sid)}{gate_marker}{rshelf_marker}")

    print(f"\n  attribution of {len(g_only)} g-only chunks:")
    print(f"    on r's shelf ({shelf_label.get(r_shelf_id, '?')!r}):       {g_only_on_r_shelf}")
    print(f"    on at least one Pass-2-gated shelf only:  {g_only_on_gated}")
    other = len(g_only) - g_only_on_r_shelf - max(0, g_only_on_gated - g_only_on_r_shelf)
    print(f"    on different (non-gated) shelves than r:  ~{len(g_only) - g_only_on_r_shelf}")

    # --- Sample excerpts: are g-only / r-only about the same topic? ------
    g_only_sample = rng.sample(list(g_only), min(N_SAMPLE_EXCERPTS, len(g_only)))
    r_only_sample = rng.sample(list(r_only), min(N_SAMPLE_EXCERPTS, len(r_only)))
    shared_sample = rng.sample(list(inter), min(N_SAMPLE_EXCERPTS, len(inter)))

    for name, sample in [("shared (in both)", shared_sample),
                         ("g-only (Pass 1 has, Pass 2 doesn't)", g_only_sample),
                         ("r-only (Pass 2 has, Pass 1 doesn't)", r_only_sample)]:
        if not sample:
            continue
        chunks = fs.chunk_store.get_many(sample)
        print(f"\n  {name}:")
        for c in chunks:
            text = (c.text or "").strip().replace("\n", " ")
            if len(text) > EXCERPT_CHARS:
                text = text[:EXCERPT_CHARS] + "…"
            cshelves_str = ",".join(sorted(shelf_label.get(s, s) for s in chunk_shelves.get(c.chunk_id, set()))[:3])
            print(f"    [{cshelves_str[:60]}]  {text}")

print(f"\n{'─' * 90}\n")

# --- Verdict heuristics ----------------------------------------------------
print("VERDICT HEURISTICS")
print(f"\nGated shelves (below min_chunks_per_shelf={min_chunks_per_shelf}): {len(gated_shelves)}")
if gated_shelves:
    sample_gated = list(gated_shelves)[:8]
    print(f"  examples: {[shelf_label.get(s, s) for s in sample_gated]}")

# How many CHUNKS in the foods facet are excluded from Pass 2 entirely?
chunks_on_at_least_one_open_shelf = 0
chunks_on_only_gated_shelves = 0
for cid, sids in chunk_shelves.items():
    if not sids:
        continue
    foods_sids = {s for s in sids if s in shelf_label}
    if not foods_sids:
        continue
    if foods_sids - gated_shelves:
        chunks_on_at_least_one_open_shelf += 1
    else:
        chunks_on_only_gated_shelves += 1

total_chunks = chunks_on_at_least_one_open_shelf + chunks_on_only_gated_shelves
if total_chunks:
    pct = 100.0 * chunks_on_only_gated_shelves / total_chunks
    print(f"\nChunks excluded from Pass 2 entirely (only on gated shelves): "
          f"{chunks_on_only_gated_shelves}/{total_chunks} ({pct:.1f}%)")
    if pct > 20:
        print("  ⚠ More than 20% of chunks Pass 1 sees never enter Pass 2. "
              "This alone can explain the disjoint catalogs.")

print("\nRead each pair's excerpts above:")
print("  - If shared / g-only / r-only chunks are all about the same topic")
print("    → the merge logic is wrong (chunks belong together but didn't merge).")
print("    Consider changing merge to asymmetric absorb (§7g option 2).")
print("  - If g-only and r-only chunks are about DIFFERENT topics that just share")
print("    some vocabulary → the passes are genuinely complementary, not redundant.")
print("    Drop the merge step entirely (§7g option 1).")


In [ ]:
# Cross-store audit + per-pass tuning canaries. Run AFTER a live build.
# `report.passed` enforces the CRITICAL invariants (parity = 1.0, no dangling
# theme_ids, no empty themes). The per-pass + merged_rate fields are WARN-level
# tuning feedback — they don't flip `passed`.

from foodscholar.layer_b.audit import audit_layer_b

report = audit_layer_b(fs.chunk_store, fs.graph_store)
print(report.model_dump_json(indent=2))

# Canary checks (won't fail the audit but worth flagging in the notebook):
if report.by_pass.get("relatedness", 0) == 0:
    print("\nWARN: 0 relatedness themes — Pass 2 found no entity-coherent clusters.")
    print("      Try cfg.layer_b.relatedness.min_shared_ids=1 or max_doc_frequency=0.6.")
if report.merged_rate >= cfg.layer_b.audit.merged_rate_max:
    print(f"\nWARN: merged_rate={report.merged_rate:.2f} is high — Pass 2 may not be earning compute.")
elif report.merged_rate <= cfg.layer_b.audit.merged_rate_min and report.merged_rate > 0:
    print(f"\nWARN: merged_rate={report.merged_rate:.2f} is low — the two passes barely overlap.")


In [ ]:
# §17 sanity gate — sample 20 random themes, print label + 3 chunk excerpts.
# Target ≥ 75% coherent by eye (label names something specific; chunks share
# that focus). Use this to decide whether the run is shippable.

import random

themes = fs.graph_store.list_themes()
print(f"total themes: {len(themes)}")

rng = random.Random(42)  # deterministic sample so audit is reproducible
sample = rng.sample(themes, min(20, len(themes)))

for i, t in enumerate(sample, 1):
    print(f"\n--- {i:>2}. [{t.discovery_pass:>11}] {t.label}")
    print(f"     id: {t.theme_id}")
    print(f"     keywords: {t.keyword_terms[:5]}")
    print(f"     foodon: {t.foodon_id_signature[:5]}")
    chunk_ids = fs.graph_store.get_chunks_for_theme(t.theme_id)
    print(f"     chunks: {len(chunk_ids)}")
    sample_chunks = fs.chunk_store.get_many(chunk_ids[:3])
    for c in sample_chunks:
        excerpt = c.text[:250].replace("\n", " ")
        print(f"       - [{c.chunk_id}] {excerpt}...")


## 8. Inspect


In [ ]:
# A handful of chunks with their attached mentions / foodon ids.
for c in fs.chunk_store.scan()[:3]:
    print(f"\n[{c.chunk_id}] {c.source_type} — {c.text[:80]}...")
    print(f"  mentions:   {[m.text for m in c.mentions[:6]]}")
    print(f"  foodon_ids: {c.foodon_ids[:6]}")
    other = sorted({ln.ontology_id for ln in c.entity_links if not ln.ontology_id.startswith('FOODON:')})
    print(f"  other_links: {other[:6]}")

# BM25 round-trip against Elastic.
hits = fs.chunk_store.search("olive oil", k=3, use_vector=False)
print("\n'olive oil' top 3:")
for h in hits:
    print(f"  {h.chunk_id}  {h.text[:80]}")

## 9. Visualize

`fs.viz` builds typed graphs (`VizGraph`) at five abstraction levels and renders
them via pluggable backends. Requires the `[viz]` extra (`pip install -e '.[viz]'`).

| Level | Method                                  | What it shows |
| ----- | --------------------------------------- | ------------- |
| L0    | `fs.viz.entity_histogram(prefix=, k=)`  | Top-k entities by chunk_count (stats) |
| L1    | `fs.viz.entity_neighborhood(ontology_id)` | One entity + mentioning chunks + co-entities |
| L2    | `fs.viz.shelf(shelf_id)`                | One Layer A shelf + themes + chunks |
| L3    | `fs.viz.backbone(facet=)`               | Whole Layer A/B/C backbone |
| L4    | `fs.viz.ontology_subtree(ontology_id)`  | FoodOn ancestors + descendants |

**Renderer choice** via `.render(backend, output=...)`:
- `"pyvis"`      → interactive HTML, **vis.js inlined** → renders inline in any
  notebook (incl. VS Code) with no network. **Use this in-notebook.**
- `"cytoscape"`  → loads cytoscape.js from a CDN; the VS Code notebook webview
  blocks that fetch, so it shows an **empty** canvas. Only use when writing to a
  standalone `.html` file opened in a real browser (`output=...`).
- `"graphviz"`   → static SVG / PNG (needs `dot` binary).
- `"matplotlib"` → bars / heatmaps over node weights (ignores edges; great for L0).

Below, the graph cells use **pyvis** (inline), and §9t prints the Layer-A
backbone as a **text tree** for a quick, dependency-free look.

In [ ]:
# L0 — top entities, rendered with matplotlib (inline figure).
fig = fs.viz.entity_histogram(k=20).render("matplotlib")

In [ ]:
# L1 — entity neighborhood, rendered with PYVIS (vis.js inlined → renders inline
# in any notebook, no CDN). cytoscape needs a CDN fetch the VS Code webview blocks,
# which is why earlier runs showed an empty box.
from IPython.display import HTML

top = fs.entities.list(prefix='FOODON', k=1)
anchor_id = top[0].ontology_id if top else 'FOODON:03309927'

html = fs.viz.entity_neighborhood(anchor_id, max_chunks=8).render('pyvis')
HTML(html)

In [ ]:
# L4 — ontology subtree around a single FoodOn id, via PYVIS (inline).
from IPython.display import HTML

# Top entities by chunk_count are often leaves in the FoodOn hierarchy (e.g.
# `food calorie datum`) — picking one of those produces a 1-edge subtree. Walk
# the top-50 and prefer anchors with real downward structure.
_onto = fs.ontology
_top = [e.ontology_id for e in fs.entities.list(prefix='FOODON', k=50)]
_resolved = [(oid, len(_onto.id_to_descendants(oid)))
             for oid in _top if _onto.get(oid) is not None]
subtree_id = next((oid for oid, n in _resolved if n >= 3),
                  next((oid for oid, n in _resolved if n >= 1),
                       _resolved[0][0] if _resolved else anchor_id))
if subtree_id != anchor_id:
    n_desc = dict(_resolved).get(subtree_id, 0)
    print(f'L4 anchor: {subtree_id} ({n_desc} descendants)')

rg_subtree = fs.viz.ontology_subtree(subtree_id, max_descendants=20)
print(rg_subtree)
HTML(rg_subtree.render('pyvis'))

In [ ]:
# L3 — Layer A backbone (foods facet), via PYVIS (inline). With bottom-up
# grouping on, the depth-1 nodes are the LLM food groups shown by their friendly
# display_label (e.g. "Fruits", "Dairy and Eggs"), not raw FoodOn labels.
from IPython.display import HTML

rg_backbone = fs.viz.backbone(facet='foods')
print(rg_backbone)
HTML(rg_backbone.render('pyvis'))

### 9t. Layer A backbone as a TEXT tree

A dependency-free, copy-pasteable view of the foods backbone — the same
shelves as the pyvis graph above, printed as an indented tree by
`display_label` with chunk counts. Useful in any terminal / for diffing builds.

In [ ]:
# Text-tree print of a Layer A facet backbone — walks shelves from the graph
# store, no viz extra / browser needed. Shows display_label (group name) +
# chunk_count; sorts children by chunk_count desc.
def print_shelf_tree(fs, facet="foods", max_depth=None, max_children=None, top=None):
    shelves = [h.model for h in fs.graph.shelves(facet=facet)]
    if not shelves:
        print(f"(no shelves for facet {facet!r})")
        return
    by_id = {s.shelf_id: s for s in shelves}
    children = {}
    for s in shelves:
        children.setdefault(s.parent_shelf_id, []).append(s)
    for kids in children.values():
        kids.sort(key=lambda s: (-s.chunk_count, (s.display_label or s.label).lower()))

    roots = children.get(None, [])
    # fall back: if parent ids point outside this facet, treat min-depth as roots
    if not roots:
        d0 = min(s.depth for s in shelves)
        roots = sorted((s for s in shelves if s.depth == d0),
                       key=lambda s: -s.chunk_count)

    n_shown = 0
    def walk(shelf, prefix, is_last, depth):
        nonlocal n_shown
        if max_depth is not None and depth > max_depth:
            return
        name = shelf.display_label or shelf.label
        kids = children.get(shelf.shelf_id, [])
        tag = f" [{len(kids)} sub]" if kids else ""
        connector = "└─ " if is_last else "├─ "
        print(f"{prefix}{connector if depth else ''}{name}  ({shelf.chunk_count:,} chunks){tag}")
        n_shown += 1
        shown = kids if max_children is None else kids[:max_children]
        child_prefix = prefix + ("   " if is_last else "│  ") if depth else ""
        for i, ch in enumerate(shown):
            walk(ch, child_prefix, i == len(shown) - 1, depth + 1)
        if max_children is not None and len(kids) > max_children:
            print(f"{child_prefix}└─ … +{len(kids) - max_children} more")

    shown_roots = roots if top is None else roots[:top]
    for i, r in enumerate(shown_roots):
        walk(r, "", i == len(shown_roots) - 1, 0)
    print(f"\n{len(shelves)} shelves total ({facet} facet)")


# Top level + first tier; cap children per node so the print stays readable.
print_shelf_tree(fs, facet="foods", max_depth=2, max_children=15)


---

## Under the hood (appendix)

The three cells above are everything an end-user needs. The appendix below is
for contributors and shows the same pipeline **without** pre-computed NER/NEL —
i.e. running GLiNER + HNSW from scratch. Skip it unless you're debugging the
NER side.

### A1. Run GLiNER + HNSW directly

Requires `pip install -e '.[annotate]'`. First call downloads the GLiNER bio
model (~1.5 GB) and the BioLORD encoder (~440 MB), then builds and caches the
FoodOn HNSW index under `data/`. Subsequent runs are fast.

In [ ]:
import importlib.util as _u

needs = [pkg for pkg in ("gliner", "hnswlib", "sentence_transformers") if not _u.find_spec(pkg)]
if needs:
    print("Skipping — missing:", needs, "\nInstall with:  pip install -e '.[annotate]'")
else:
    fs_g = FoodScholar.from_config({
        "corpus": {"chunks_path": str(CORPUS_DIR)},
        "ontology": {
            "foodon_path": str(FOODON_OWL),
            "cache_path": str(REPO_ROOT / "data" / "foodon_cache.parquet"),
            "prefix_filter": ["FOODON:"],
        },
        "storage": {
            "chunk_store": {"backend": "memory"},
            "graph_store": {"backend": "memory"},
        },
    })
    one_file = next(iter(sorted(CORPUS_DIR.glob("*.csv"))))
    fs_g.ingest(one_file)  # no nel_dir → GLiNER + HNSW path
    c = fs_g.chunk_store.scan()[0]
    print(c.chunk_id, "mentions:", [m.text for m in c.mentions[:6]])

### A2. Inspect the graph

Layer B/C builders are still stubs — this appendix shows the `fs.graph` API
the builders will write through. Run it against the in-memory facade so it
doesn't depend on the live services.

In [ ]:
fs_local = FoodScholar.in_memory()
fs_local.graph.add_shelf(shelf_id="s-med", label="Mediterranean diet",
                         facet="dietary_patterns", depth=0)
fs_local.graph.add_theme(theme_id="t-olive", label="Olive oil benefits",
                         shelf_ids=["s-med"], discovered_by="leiden",
                         discovery_version="nb")
shelf = fs_local.graph.shelf("s-med")
print("shelf :", shelf.label, "| facet:", shelf.facet)
print("themes:", [t.label for t in shelf.themes()])